<a href="https://colab.research.google.com/github/filfish2007-cav/Hospital_LLM/blob/main/DataCleaning_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installing pandas and matplotlib


In [2]:
import pandas as pd
import matplotlib as plt

### Exploratory Data Analysis: Dataset Overview
Inspect the structure, columns, data types, and first few rows of each of the 6 DataFrames.

In [3]:
# =========================
# 1. LOAD DATA
# =========================

base_url = "https://raw.githubusercontent.com/filfish2007-cav/Hospital_LLM/main/data/hospital_csvs/"

files = [
    "patients.csv",
    "encounters.csv",
    "conditions.csv",
    "medications.csv",
    "observations.csv",
    "procedures.csv"
]

data = {
    file.replace(".csv", ""): pd.read_csv(base_url + file)
    for file in files
}


# =========================
# 2. BASIC INSPECTION
# =========================

for name, df in data.items():

    print("\n" + "=" * 80)
    print(f"TABLE: {name.upper()}")
    print("=" * 80)

    # 1. Number of rows / columns
    print("\n1. Shape")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    # 2. Column names
    print("\n2. Column names")
    print(df.columns.tolist())

    # 3. Data types
    print("\n3. Data types")
    print(df.dtypes)

    # 4. Missing values
    print("\n4. Missing values")
    missing = df.isnull().sum()
    missing_percentage = (df.isnull().mean() * 100).round(2)

    missing_report = pd.DataFrame({
        "missing_count": missing,
        "missing_percentage": missing_percentage
    })

    print(missing_report[missing_report["missing_count"] > 0])

    if missing.sum() == 0:
        print("No missing values.")

    # 5. Duplicated rows
    print("\n5. Duplicated rows")
    duplicates = df.duplicated().sum()
    print(f"Duplicate rows: {duplicates}")

    # 6. Unique IDs
    print("\n6. Potential ID columns")

    id_columns = [
        column for column in df.columns
        if column.lower() == "id"
        or column.lower().endswith("_id")
        or column.lower() == "patient"
        or column.lower() == "encounter"
    ]

    if id_columns:
        for column in id_columns:
            print(
                f"{column}: "
                f"{df[column].nunique()} unique / "
                f"{len(df)} total"
            )
        else:
            print("No obvious ID columns found.")

    # 7. Suspicious numeric values
    print("\n7. Numeric value overview")

    numeric_columns = df.select_dtypes(include="number").columns

    if len(numeric_columns) > 0:
        print(df[numeric_columns].describe().T)
    else:
        print("No numeric columns.")

    # 8. Date ranges
    print("\n8. Date columns / ranges")

    date_columns = []

    for column in df.columns:
        if (
            "date" in column.lower()
            or "time" in column.lower()
            or "birth" in column.lower()
            or "death" in column.lower()
        ):
            date_columns.append(column)

    if date_columns:
        for column in date_columns:

            dates = pd.to_datetime(
                df[column],
                errors="coerce"
            )

            print(
                f"{column}: "
                f"{dates.min()} → {dates.max()}"
            )
    else:
        print("No obvious date columns found.")

    # 9. Categorical values
    print("\n9. Categorical columns")

    categorical_columns = df.select_dtypes(
        include=["object", "category"]
    ).columns

    for column in categorical_columns:

        unique_values = df[column].dropna().unique()

        print(f"\n{column}")
        print(f"Unique values: {len(unique_values)}")

        # Print values only when there aren't too many
        if len(unique_values) <= 30:
            print(unique_values)
        else:
            print("Too many unique values to display.")


# =========================
# 3. RELATIONSHIPS BETWEEN TABLES
# =========================

print("\n" + "=" * 80)
print("10. RELATIONSHIPS BETWEEN TABLES")
print("=" * 80)


# Collect columns from every table
all_columns = {}

for table_name, df in data.items():
    for column in df.columns:
        all_columns.setdefault(column, []).append(table_name)


print("\nColumns appearing in multiple tables:")

for column, tables in all_columns.items():

    if len(tables) > 1:
        print(
            f"{column}: "
            f"{', '.join(tables)}"
        )


# =========================
# 4. FOREIGN KEY CHECKS
# =========================

print("\n" + "=" * 80)
print("FOREIGN KEY / ID INTEGRITY CHECKS")
print("=" * 80)


# Common relationships expected in Synthea
relationships = [
    ("encounters", "PATIENT", "patients", "Id"),
    ("conditions", "PATIENT", "patients", "Id"),
    ("conditions", "ENCOUNTER", "encounters", "Id"),
    ("medications", "PATIENT", "patients", "Id"),
    ("medications", "ENCOUNTER", "encounters", "Id"),
    ("observations", "PATIENT", "patients", "Id"),
    ("observations", "ENCOUNTER", "encounters", "Id"),
    ("procedures", "PATIENT", "patients", "Id"),
    ("procedures", "ENCOUNTER", "encounters", "Id"),
]


for child_table, child_column, parent_table, parent_column in relationships:

    if (
        child_table in data
        and parent_table in data
        and child_column in data[child_table].columns
        and parent_column in data[parent_table].columns
    ):

        child_ids = set(
            data[child_table][child_column]
            .dropna()
            .unique()
        )

        parent_ids = set(
            data[parent_table][parent_column]
            .dropna()
            .unique()
        )

        orphan_ids = child_ids - parent_ids

        print(
            f"\n{child_table}.{child_column}"
            f" → "
            f"{parent_table}.{parent_column}"
        )

        print(f"Referenced IDs: {len(child_ids)}")
        print(f"Missing parent IDs: {len(orphan_ids)}")

        if orphan_ids:
            print("WARNING: orphan IDs detected!")
        else:
            print("OK: all references exist.")


TABLE: PATIENTS

1. Shape
Rows: 120
Columns: 28

2. Column names
['Id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE', 'INCOME']

3. Data types
Id                      object
BIRTHDATE               object
DEATHDATE               object
SSN                     object
DRIVERS                 object
PASSPORT                object
PREFIX                  object
FIRST                   object
MIDDLE                  object
LAST                    object
SUFFIX                  object
MAIDEN                  object
MARITAL                 object
RACE                    object
ETHNICITY               object
GENDER                  object
BIRTHPLACE              object
ADDRESS                 object
CITY                    object
STATE                   object


/tmp/ipykernel_8750/1251747611.py:113: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(



SYSTEM
Unique values: 1
['http://snomed.info/sct']

DESCRIPTION
Unique values: 185
Too many unique values to display.

TABLE: MEDICATIONS

1. Shape
Rows: 8001
Columns: 13

2. Column names
['START', 'STOP', 'PATIENT', 'PAYER', 'ENCOUNTER', 'CODE', 'DESCRIPTION', 'BASE_COST', 'PAYER_COVERAGE', 'DISPENSES', 'TOTALCOST', 'REASONCODE', 'REASONDESCRIPTION']

3. Data types
START                 object
STOP                  object
PATIENT               object
PAYER                 object
ENCOUNTER             object
CODE                   int64
DESCRIPTION           object
BASE_COST            float64
PAYER_COVERAGE       float64
DISPENSES              int64
TOTALCOST            float64
REASONCODE           float64
REASONDESCRIPTION     object
dtype: object

4. Missing values
                   missing_count  missing_percentage
STOP                         348                4.35
REASONCODE                  1047               13.09
REASONDESCRIPTION           1047               13.09

5. Dupl

### Obserations: Exploring duplicates (22 rows) in observation column

In [4]:
duplicates = data["observations"][
    data["observations"].duplicated(keep=False)
]

display(duplicates)

,DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE
5622,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2339-0,Glucose [Mass/volume] in Blood,90.7,mg/dL,numeric
5623,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,6299-2,Urea nitrogen [Mass/volume] in Blood,17.4,mg/dL,numeric
5625,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,49765-1,Calcium [Mass/volume] in Blood,9.5,mg/dL,numeric
5626,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2947-0,Sodium [Moles/volume] in Blood,136.2,mmol/L,numeric
5627,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,6298-4,Potassium [Moles/volume] in Blood,3.8,mmol/L,numeric
5628,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2069-3,Chloride [Moles/volume] in Blood,109.1,mmol/L,numeric
5629,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,20565-8,Carbon dioxide total [Moles/volume] in Blood,23.1,mmol/L,numeric
5646,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2339-0,Glucose [Mass/volume] in Blood,90.7,mg/dL,numeric
5647,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,6299-2,Urea nitrogen [Mass/volume] in Blood,17.4,mg/dL,numeric
5649,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,49765-1,Calcium [Mass/volume] in Blood,9.5,mg/dL,numeric


In [5]:
print(f"Total rows in the duplicates DataFrame (all occurrences, keep=False): {len(duplicates)}")
print(f"Redundant duplicate rows (keep='first'): {data['observations'].duplicated(keep='first').sum()}")

Total rows in the duplicates DataFrame (all occurrences, keep=False): 44
Redundant duplicate rows (keep='first'): 22


In [6]:
# Sort duplicates to place identical pairs next to each other
sorted_duplicates = duplicates.sort_values(by=["PATIENT", "DATE", "CODE"])
display(sorted_duplicates)

,DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE
5629,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,20565-8,Carbon dioxide total [Moles/volume] in Blood,23.1,mmol/L,numeric
5653,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,20565-8,Carbon dioxide total [Moles/volume] in Blood,23.1,mmol/L,numeric
5628,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2069-3,Chloride [Moles/volume] in Blood,109.1,mmol/L,numeric
5652,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2069-3,Chloride [Moles/volume] in Blood,109.1,mmol/L,numeric
5622,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2339-0,Glucose [Mass/volume] in Blood,90.7,mg/dL,numeric
5646,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2339-0,Glucose [Mass/volume] in Blood,90.7,mg/dL,numeric
5626,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2947-0,Sodium [Moles/volume] in Blood,136.2,mmol/L,numeric
5650,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,2947-0,Sodium [Moles/volume] in Blood,136.2,mmol/L,numeric
5625,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,49765-1,Calcium [Mass/volume] in Blood,9.5,mg/dL,numeric
5649,2019-07-28T00:34:27Z,4b732c6e-10df-885f-1d17-253050ced633,4b732c6e-10df-885f-afbd-b977f92e9340,laboratory,49765-1,Calcium [Mass/volume] in Blood,9.5,mg/dL,numeric


In [7]:
# 1. Check if 'observations' has a primary key ID used in other tables
# Usually observations don't have an ID column in this dataset, let's verify.
print("Columns in observations:", data["observations"].columns.tolist())

# 2. Check if removing duplicates from observations alters unique patient/encounter values
original_patients = data["observations"]["PATIENT"].nunique()
original_encounters = data["observations"]["ENCOUNTER"].nunique()

clean_observations = data["observations"].drop_duplicates()

clean_patients = clean_observations["PATIENT"].nunique()
clean_encounters = clean_observations["ENCOUNTER"].nunique()

print(f"Patients before: {original_patients} | After: {clean_patients}")
print(f"Encounters before: {original_encounters} | After: {clean_encounters}")

Columns in observations: ['DATE', 'PATIENT', 'ENCOUNTER', 'CATEGORY', 'CODE', 'DESCRIPTION', 'VALUE', 'UNITS', 'TYPE']
Patients before: 120 | After: 120
Encounters before: 4018 | After: 4018


### Observations: Deleting duplicates since it does not affect other tables

In [8]:
# 1. Create a safe standalone backup variable outside the data dictionary
observations_backup = data["observations"].copy()
if "observations_backup" in data:
    del data["observations_backup"]

# 2. Drop duplicates in place from the active observations table
original_row_count = len(data["observations"])
data["observations"] = data["observations"].drop_duplicates()
new_row_count = len(data["observations"])

print("Backup created as standalone variable 'observations_backup' (removed from 'data' dictionary).")
print(f"Active table deduplicated: {original_row_count} rows -> {new_row_count} rows (removed {original_row_count - new_row_count} duplicate rows).")

Backup created as standalone variable 'observations_backup' (removed from 'data' dictionary).
Active table deduplicated: 117524 rows -> 117502 rows (removed 22 duplicate rows).


In [9]:
# 1. Ensure the backup key is completely removed from the data dictionary
if "observations_backup" in data:
    del data["observations_backup"]

# 2. Verify that the standalone backup variable still exists and is safe in memory
try:
    print(f"✓ Standalone 'observations_backup' is available!")
    print(f"  Shape: {observations_backup.shape}")
    print(f"  Active 'data' dictionary keys: {list(data.keys())}")
except NameError:
    print("❌ Standalone backup was not found. Re-creating it now...")
    observations_backup = data["observations"].copy()
    print("✓ Standalone 'observations_backup' created successfully!")

✓ Standalone 'observations_backup' is available!
  Shape: (117524, 9)
  Active 'data' dictionary keys: ['patients', 'encounters', 'conditions', 'medications', 'observations', 'procedures']


In [10]:
for name, df in data.items():

    print("\n" + "=" * 80)
    print(f"TABLE: {name.upper()}")
    print("=" * 80)

    # 1. Number of rows / columns
    print("\n1. Shape")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    # 2. Column names
    print("\n2. Column names")
    print(df.columns.tolist())

    # 3. Data types
    print("\n3. Data types")
    print(df.dtypes)

    # 4. Missing values
    print("\n4. Missing values")
    missing = df.isnull().sum()
    missing_percentage = (df.isnull().mean() * 100).round(2)

    missing_report = pd.DataFrame({
        "missing_count": missing,
        "missing_percentage": missing_percentage
    })

    print(missing_report[missing_report["missing_count"] > 0])

    if missing.sum() == 0:
        print("No missing values.")

    # 5. Duplicated rows
    print("\n5. Duplicated rows")
    duplicates = df.duplicated().sum()
    print(f"Duplicate rows: {duplicates}")

    # 6. Unique IDs
    print("\n6. Potential ID columns")

    id_columns = [
        column for column in df.columns
        if column.lower() == "id"
        or column.lower().endswith("_id")
        or column.lower() == "patient"
        or column.lower() == "encounter"
    ]

    if id_columns:
        for column in id_columns:
            print(
                f"{column}: "
                f"{df[column].nunique()} unique / "
                f"{len(df)} total"
            )
        else:
            print("No obvious ID columns found.")

    # 7. Suspicious numeric values
    print("\n7. Numeric value overview")

    numeric_columns = df.select_dtypes(include="number").columns

    if len(numeric_columns) > 0:
        print(df[numeric_columns].describe().T)
    else:
        print("No numeric columns.")

    # 8. Date ranges
    print("\n8. Date columns / ranges")

    date_columns = []

    for column in df.columns:
        if (
            "date" in column.lower()
            or "time" in column.lower()
            or "birth" in column.lower()
            or "death" in column.lower()
        ):
            date_columns.append(column)

    if date_columns:
        for column in date_columns:

            dates = pd.to_datetime(
                df[column],
                errors="coerce"
            )

            print(
                f"{column}: "
                f"{dates.min()} → {dates.max()}"
            )
    else:
        print("No obvious date columns found.")

    # 9. Categorical values
    print("\n9. Categorical columns")

    categorical_columns = df.select_dtypes(
        include=["object", "category"]
    ).columns

    for column in categorical_columns:

        unique_values = df[column].dropna().unique()

        print(f"\n{column}")
        print(f"Unique values: {len(unique_values)}")

        # Print values only when there aren't too many
        if len(unique_values) <= 30:
            print(unique_values)
        else:
            print("Too many unique values to display.")


# =========================
# 3. RELATIONSHIPS BETWEEN TABLES
# =========================

print("\n" + "=" * 80)
print("10. RELATIONSHIPS BETWEEN TABLES")
print("=" * 80)


# Collect columns from every table
all_columns = {}

for table_name, df in data.items():
    for column in df.columns:
        all_columns.setdefault(column, []).append(table_name)


print("\nColumns appearing in multiple tables:")

for column, tables in all_columns.items():

    if len(tables) > 1:
        print(
            f"{column}: "
            f"{', '.join(tables)}"
        )


# =========================
# 4. FOREIGN KEY CHECKS
# =========================

print("\n" + "=" * 80)
print("FOREIGN KEY / ID INTEGRITY CHECKS")
print("=" * 80)


# Common relationships expected in Synthea
relationships = [
    ("encounters", "PATIENT", "patients", "Id"),
    ("conditions", "PATIENT", "patients", "Id"),
    ("conditions", "ENCOUNTER", "encounters", "Id"),
    ("medications", "PATIENT", "patients", "Id"),
    ("medications", "ENCOUNTER", "encounters", "Id"),
    ("observations", "PATIENT", "patients", "Id"),
    ("observations", "ENCOUNTER", "encounters", "Id"),
    ("procedures", "PATIENT", "patients", "Id"),
    ("procedures", "ENCOUNTER", "encounters", "Id"),
]


for child_table, child_column, parent_table, parent_column in relationships:

    if (
        child_table in data
        and parent_table in data
        and child_column in data[child_table].columns
        and parent_column in data[parent_table].columns
    ):

        child_ids = set(
            data[child_table][child_column]
            .dropna()
            .unique()
        )

        parent_ids = set(
            data[parent_table][parent_column]
            .dropna()
            .unique()
        )

        orphan_ids = child_ids - parent_ids

        print(
            f"\n{child_table}.{child_column}"
            f" → "
            f"{parent_table}.{parent_column}"
        )

        print(f"Referenced IDs: {len(child_ids)}")
        print(f"Missing parent IDs: {len(orphan_ids)}")

        if orphan_ids:
            print("WARNING: orphan IDs detected!")
        else:
            print("OK: all references exist.")


TABLE: PATIENTS

1. Shape
Rows: 120
Columns: 28

2. Column names
['Id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE', 'INCOME']

3. Data types
Id                      object
BIRTHDATE               object
DEATHDATE               object
SSN                     object
DRIVERS                 object
PASSPORT                object
PREFIX                  object
FIRST                   object
MIDDLE                  object
LAST                    object
SUFFIX                  object
MAIDEN                  object
MARITAL                 object
RACE                    object
ETHNICITY               object
GENDER                  object
BIRTHPLACE              object
ADDRESS                 object
CITY                    object
STATE                   object


/tmp/ipykernel_8750/677355841.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(




5. Duplicated rows
Duplicate rows: 0

6. Potential ID columns
PATIENT: 120 unique / 117502 total
ENCOUNTER: 4018 unique / 117502 total
No obvious ID columns found.

7. Numeric value overview
No numeric columns.

8. Date columns / ranges
DATE: 1967-07-01 01:30:03+00:00 → 2026-09-23 13:15:05+00:00

9. Categorical columns

DATE
Unique values: 10929
Too many unique values to display.

PATIENT
Unique values: 120
Too many unique values to display.

ENCOUNTER
Unique values: 4018
Too many unique values to display.

CATEGORY
Unique values: 8
['vital-signs' 'laboratory' 'social-history' 'survey' 'exam' 'procedure'
 'therapy' 'imaging']

CODE
Unique values: 241
Too many unique values to display.

DESCRIPTION
Unique values: 242
Too many unique values to display.

VALUE
Unique values: 3583
Too many unique values to display.

UNITS
Unique values: 46
Too many unique values to display.

TYPE
Unique values: 2
['numeric' 'text']

TABLE: PROCEDURES

1. Shape
Rows: 21901
Columns: 10

2. Column names
['S

### All: Standardising column names

In [11]:
# Inspect the exact column names of all loaded tables to identify naming discrepancies
for name, df in data.items():
    columns = df.columns.tolist()
    # Check for any leading/trailing spaces in column names
    spaced_columns = [col for col in columns if col != col.strip()]

    print(f"Table: {name.upper()}")
    print(f"  Columns: {columns}")
    if spaced_columns:
        print(f"  ⚠️ WARNING: Columns with hidden spaces: {spaced_columns}")
    else:
        print("  ✓ No hidden spaces in column names.")
    print("-" * 50)

Table: PATIENTS
  Columns: ['Id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE', 'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE', 'INCOME']
  ✓ No hidden spaces in column names.
--------------------------------------------------
Table: ENCOUNTERS
  Columns: ['Id', 'START', 'STOP', 'PATIENT', 'ORGANIZATION', 'PROVIDER', 'PAYER', 'ENCOUNTERCLASS', 'CODE', 'DESCRIPTION', 'BASE_ENCOUNTER_COST', 'TOTAL_CLAIM_COST', 'PAYER_COVERAGE', 'REASONCODE', 'REASONDESCRIPTION']
  ✓ No hidden spaces in column names.
--------------------------------------------------
Table: CONDITIONS
  Columns: ['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'SYSTEM', 'CODE', 'DESCRIPTION']
  ✓ No hidden spaces in column names.
--------------------------------------------------
Table: MEDICATIONS
  Columns: ['START', 'STOP', 'PATIENT', 'PAYER

In [12]:
import re

def to_snake_case(name):
    # Handle common camelCase or PascalCase split points first
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    s2 = re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()
    # Replace potential double underscores and clean up
    return re.sub('_+', '_', s2)

# Standardize column headers for each active table in 'data'
for table_name, df in data.items():

    # Apply snake_case transformation
    new_columns = [to_snake_case(col) for col in df.columns]

    # Specifically handle standard ID columns we want as simple 'id'
    new_columns = ['id' if col == 'id' else col for col in new_columns]

    df.columns = new_columns
    print(f"Table '{table_name}' updated columns:")
    print(df.columns.tolist())
    print("-" * 50)

Table 'patients' updated columns:
['id', 'birthdate', 'deathdate', 'ssn', 'drivers', 'passport', 'prefix', 'first', 'middle', 'last', 'suffix', 'maiden', 'marital', 'race', 'ethnicity', 'gender', 'birthplace', 'address', 'city', 'state', 'county', 'fips', 'zip', 'lat', 'lon', 'healthcare_expenses', 'healthcare_coverage', 'income']
--------------------------------------------------
Table 'encounters' updated columns:
['id', 'start', 'stop', 'patient', 'organization', 'provider', 'payer', 'encounterclass', 'code', 'description', 'base_encounter_cost', 'total_claim_cost', 'payer_coverage', 'reasoncode', 'reasondescription']
--------------------------------------------------
Table 'conditions' updated columns:
['start', 'stop', 'patient', 'encounter', 'system', 'code', 'description']
--------------------------------------------------
Table 'medications' updated columns:
['start', 'stop', 'patient', 'payer', 'encounter', 'code', 'description', 'base_cost', 'payer_coverage', 'dispenses', 'to

### Patiens: checking ids

In [13]:
import re

patients_df = data['patients']

# 1. Check for Nulls
null_count = patients_df['id'].isnull().sum()
print(f"Null IDs count: {null_count}")

# 2. Check for Whitespace and Clean in-place
patients_df['id'] = patients_df['id'].astype(str).str.strip()
print("Strip complete. Whitespace has been removed from 'id' column.")

# 3. Check for uniqueness
unique_ids = patients_df['id'].nunique()
total_rows = len(patients_df)
print(f"Unique IDs: {unique_ids} | Total Rows: {total_rows}")
if unique_ids == total_rows:
    print("✓ All patient IDs are completely unique.")
else:
    print("⚠️ WARNING: Duplicate patient IDs detected!")

# 4. Check UUID format (8-4-4-4-12 hex characters)
uuid_pattern = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$', re.IGNORECASE)
valid_uuid_mask = patients_df['id'].apply(lambda x: bool(uuid_pattern.match(x)))
invalid_uuids = patients_df[~valid_uuid_mask]['id']

print(f"Valid UUID format count: {valid_uuid_mask.sum()} / {total_rows}")
if not invalid_uuids.empty:
    print(f"⚠️ WARNING: Found {len(invalid_uuids)} IDs with an invalid format:")
    print(invalid_uuids.head())
else:
    print("✓ All patient IDs conform to a standard UUID format.")

Null IDs count: 0
Strip complete. Whitespace has been removed from 'id' column.
Unique IDs: 120 | Total Rows: 120
✓ All patient IDs are completely unique.
Valid UUID format count: 120 / 120
✓ All patient IDs conform to a standard UUID format.


### Patients - Checking Dates

In [14]:
import pandas as pd

# Safely parse dates to datetime
patients_df = data['patients']
patients_df['birthdate'] = pd.to_datetime(patients_df['birthdate'], errors='coerce')
patients_df['deathdate'] = pd.to_datetime(patients_df['deathdate'], errors='coerce')

print("1. Basic Null Checks:")
print(f"Parsed birthdates missing: {patients_df['birthdate'].isna().sum()}")
print(f"Parsed deathdates missing: {patients_df['deathdate'].isna().sum()} (out of {len(patients_df)})")

1. Basic Null Checks:
Parsed birthdates missing: 0
Parsed deathdates missing: 100 (out of 120)


In [15]:
# 2. logical check: deathdate >= birthdate
invalid_death_order = patients_df[patients_df['deathdate'] < patients_df['birthdate']]
print(f"\n2. Logical check: Patients with deathdate < birthdate: {len(invalid_death_order)}")
if not invalid_death_order.empty:
    display(invalid_death_order[['id', 'birthdate', 'deathdate']])


2. Logical check: Patients with deathdate < birthdate: 0


In [16]:
# 3. Check for future dates relative to current logic (e.g. up to 2026 as observed in date ranges)
reference_date = pd.to_datetime('2026-12-31')
future_births = patients_df[patients_df['birthdate'] > reference_date]
print(f"\n3. Future birthdates check (after 2026-12-31): {len(future_births)}")

# Calculate approximate age
patients_df['age_at_reference'] = (reference_date - patients_df['birthdate']).dt.days / 365.25
extremely_old = patients_df[patients_df['age_at_reference'] > 110]
print(f"Patients older than 110 years (relative to 2026): {len(extremely_old)}")
if not extremely_old.empty:
    display(extremely_old[['id', 'birthdate', 'age_at_reference']])


3. Future birthdates check (after 2026-12-31): 0
Patients older than 110 years (relative to 2026): 0


In [17]:
# 4. Cross-table consistency: Events before birthdate or after deathdate
# Parse dates in encounters and observations for precise comparisons
encounters_df = data['encounters'].copy()
encounters_df['start'] = pd.to_datetime(encounters_df['start'], errors='coerce').dt.tz_localize(None)
encounters_df['stop'] = pd.to_datetime(encounters_df['stop'], errors='coerce').dt.tz_localize(None)

observations_df = data['observations'].copy()
observations_df['date'] = pd.to_datetime(observations_df['date'], errors='coerce').dt.tz_localize(None)

# Align patients_df timezone
p_temp = patients_df.copy()
p_temp['birthdate'] = p_temp['birthdate'].dt.tz_localize(None)
p_temp['deathdate'] = p_temp['deathdate'].dt.tz_localize(None)

# Merge patient boundaries with clinical events
enc_check = encounters_df.merge(p_temp[['id', 'birthdate', 'deathdate']], left_on='patient', right_on='id')
obs_check = observations_df.merge(p_temp[['id', 'birthdate', 'deathdate']], left_on='patient', right_on='id')

# Check encounters/observations occurring before patient birth
enc_before_birth = enc_check[enc_check['start'] < enc_check['birthdate']]
obs_before_birth = obs_check[obs_check['date'] < obs_check['birthdate']]

print(f"\n4. Timeline anomalies (Before Birth):")
print(f"Encounters starting before birth: {len(enc_before_birth)}")
print(f"Observations recorded before birth: {len(obs_before_birth)}")

if not enc_before_birth.empty:
    print("Sample of encounters before birth:")
    display(enc_before_birth[['patient', 'birthdate', 'id_x', 'start', 'description']].head())

# Check encounters/observations occurring after patient death
enc_after_death = enc_check[(enc_check['deathdate'].notna()) & (enc_check['stop'] > enc_check['deathdate'])]
obs_after_death = obs_check[(obs_check['deathdate'].notna()) & (obs_check['date'] > obs_check['deathdate'])]

print(f"\n5. Timeline anomalies (After Death):")
print(f"Encounters finishing after death: {len(enc_after_death)}")
print(f"Observations recorded after death: {len(obs_after_death)}")

if not enc_after_death.empty:
    print("Sample of encounters after death:")
    display(enc_after_death[['patient', 'deathdate', 'id_x', 'stop', 'description']].head())


4. Timeline anomalies (Before Birth):
Encounters starting before birth: 4
Observations recorded before birth: 97
Sample of encounters before birth:


,patient,birthdate,id_x,start,description
0,f19b9cae-0ea7-2d51-f2c5-946d448992a5,2020-01-15,f19b9cae-0ea7-2d51-66f5-08c20377c8bc,2020-01-14 22:59:24,Well child visit (procedure)
143,13a674ff-e2f7-06ad-461b-e4087dcedbd4,2021-04-05,13a674ff-e2f7-06ad-ed74-af311c8b37a6,2021-04-04 22:52:49,Well child visit (procedure)
1088,f4330273-be0c-2edf-2b4e-78bd87b98ee9,2020-04-03,f4330273-be0c-2edf-8de7-09376c61b04c,2020-04-02 21:37:39,Well child visit (procedure)
2258,079614f4-526c-c3eb-a0ef-3ad3d1191236,2005-09-19,079614f4-526c-c3eb-1acd-f16e0bf67397,2005-09-18 22:42:40,Well child visit (procedure)



5. Timeline anomalies (After Death):
Encounters finishing after death: 31
Observations recorded after death: 79
Sample of encounters after death:


,patient,deathdate,id_x,stop,description
324,eec13363-c6e9-a26d-3750-814976e54a24,2013-04-06,eec13363-c6e9-a26d-528c-ebe26eb1e054,2013-04-14 07:08:34,Death Certification
540,5e816607-ff3f-8bc7-7f53-8f272a88f9a4,1970-07-24,5e816607-ff3f-8bc7-0a8f-1ba65661779d,1970-07-24 07:04:32,Admission to hospice (procedure)
555,5e816607-ff3f-8bc7-7f53-8f272a88f9a4,1970-07-24,5e816607-ff3f-8bc7-e4bd-8c5d1f9760eb,1970-07-25 01:45:03,Death Certification
1056,381d4a6a-695b-2f73-a97c-5a14b9f19608,2025-09-02,381d4a6a-695b-2f73-556c-0c24083525b6,2025-09-02 03:12:27,Admission to intensive care unit (procedure)
1057,381d4a6a-695b-2f73-a97c-5a14b9f19608,2025-09-02,381d4a6a-695b-2f73-7833-e7f8714e0ef8,2025-09-13 01:45:03,Death Certification


### Patients: Fixing anomalies(Clinical Events Recorded BEFORE Birth, Clinical Events Recorded AFTER Death)

In [18]:
tol = pd.Timedelta(days=1)

def to_naive_utc(s):
    return pd.to_datetime(s, utc=True, errors='coerce').dt.tz_localize(None)

p = data['patients'][['id', 'birthdate', 'deathdate']].copy()
p['birthdate'] = pd.to_datetime(p['birthdate']).dt.tz_localize(None)
p['deathdate'] = pd.to_datetime(p['deathdate']).dt.tz_localize(None)

enc = data['encounters'].copy()
enc['start'] = to_naive_utc(enc['start'])
enc = enc.merge(p, left_on='patient', right_on='id', suffixes=('', '_p'))

obs = data['observations'].copy()
obs['date'] = to_naive_utc(obs['date'])
obs = obs.merge(p, left_on='patient', right_on='id', suffixes=('', '_p'))

enc_before = enc[enc['start'] < enc['birthdate'] - tol]
enc_after  = enc[enc['deathdate'].notna() & (enc['start'] > enc['deathdate'] + tol)]
obs_before = obs[obs['date'] < obs['birthdate'] - tol]
obs_after  = obs[obs['deathdate'].notna() & (obs['date'] > obs['deathdate'] + tol)]





In [19]:
print("=== TIMELINE ANOMALIES AUDIT (with 1-Day Tolerance) ===")
print(f"Encounters starting > 1 day before birth: {len(enc_before)}")
print(f"Encounters starting > 1 day after death:  {len(enc_after)}")
print(f"Observations recorded > 1 day before birth: {len(obs_before)}")
print(f"Observations recorded > 1 day after death:  {len(obs_after)}")


=== TIMELINE ANOMALIES AUDIT (with 1-Day Tolerance) ===
Encounters starting > 1 day before birth: 0
Encounters starting > 1 day after death:  18
Observations recorded > 1 day before birth: 0
Observations recorded > 1 day after death:  21


### Patients: Timeline anomalies: events before birth

4 encounters and 97 observations appeared before the birthdate. All are newborn "Well child visit" encounters starting a few hours before midnight UTC on the day before the birthdate. The cause is date-only birthdates compared against UTC timestamps, not a data error.

# With a 1-day tolerance, 0 remain. **No rows removed or modified.**

In [20]:
# What kinds of encounters are they?
print(enc_after['description'].value_counts())

# How far after death, and which patients?
enc_after = enc_after.assign(days_after=(enc_after['start'] - enc_after['deathdate']).dt.days)
display(enc_after[['patient', 'deathdate', 'start', 'days_after', 'description']]
        .sort_values('days_after', ascending=False))

# Are the observations tied to those same encounters?
print(obs_after['encounter'].isin(enc_after['id']).mean())
print(obs_after['description'].value_counts().head(10))

description
Death Certification    18
Name: count, dtype: int64


,patient,deathdate,start,days_after,description
4819,7d4e9f28-6f5a-edb5-2778-cecd6efc4e9a,2021-09-23,2021-10-07 03:16:31,14,Death Certification
4404,1937dc8b-92c6-408d-ed95-ed2c384446a2,2010-05-22,2010-06-03 05:01:28,12,Death Certification
1057,381d4a6a-695b-2f73-a97c-5a14b9f19608,2025-09-02,2025-09-13 01:30:03,11,Death Certification
1472,9ae2ef41-6004-7e78-7001-1fe4da06ec90,2026-06-18,2026-06-28 19:28:17,10,Death Certification
3736,6f353dad-c5cc-3cdd-87af-fe7eca4dc35b,2015-05-12,2015-05-21 05:01:28,9,Death Certification
3929,f45ba14c-644d-c1fb-1239-fee274e0abb5,1997-10-07,1997-10-16 05:01:28,9,Death Certification
3576,64cc597f-de2d-b26f-a303-8b8beda99395,2022-08-06,2022-08-14 00:05:05,8,Death Certification
324,eec13363-c6e9-a26d-3750-814976e54a24,2013-04-06,2013-04-14 06:53:34,8,Death Certification
6135,a6b49352-759a-1fa6-d169-3f8dd7471305,2000-02-03,2000-02-08 22:08:49,5,Death Certification
5146,5dfaf183-aff9-c886-72ef-492c34861077,2006-11-02,2006-11-07 22:08:49,5,Death Certification


0.8571428571428571
description
Cause of Death [US Standard Certificate of Death]    18
QALY                                                  1
DALY                                                  1
QOLS                                                  1
Name: count, dtype: int64


In [21]:
# Flag death-related administrative events instead of removing them
death_admin_ids = enc_after['id']
data['encounters']['is_death_certification'] = (
    data['encounters']['description'].str.contains('Death Certification', na=False)
)
data['observations']['is_cause_of_death'] = (
    data['observations']['description'].str.contains('Cause of Death', na=False)
)

### Patients: Timeline anomalies: events after death

18 encounters and 21 observations fall more than 1 day after the death date. All are expected administrative records:
- **Encounters (18):** all "Death Certification", filed 1-14 days after death.
- **Observations (21):** 18 "Cause of Death" (linked to those encounters) + 3 summary metrics (QALY, DALY, QOLS).

No clinical care occurs after death. **No rows removed**; added flags `is_death_certification` and `is_cause_of_death`.

In [22]:
p = data['patients']
cols = ['gender', 'marital', 'race', 'ethnicity', 'prefix', 'suffix']

for c in cols:
    s = p[c]
    v = s.dropna().astype(str)
    print(f"\n{c}  (missing: {s.isna().sum()})")
    print(s.value_counts(dropna=False).to_dict())
    print(f"  whitespace issues: {(v != v.str.strip()).sum()} | "
          f"empty strings: {(v.str.strip() == '').sum()} | "
          f"case variants: {v.str.lower().nunique() != v.nunique()}")

# Consistency: prefix vs gender, missing marital/prefix vs age
print(pd.crosstab(p['prefix'].fillna('NA'), p['gender']))
age = (pd.Timestamp('2026-09-30') - pd.to_datetime(p['birthdate'])).dt.days / 365.25
print(age.groupby(p['marital'].isna()).median())


gender  (missing: 0)
{'F': 63, 'M': 57}
  whitespace issues: 0 | empty strings: 0 | case variants: False

marital  (missing: 39)
{'M': 47, nan: 39, 'S': 19, 'D': 12, 'W': 3}
  whitespace issues: 0 | empty strings: 0 | case variants: False

race  (missing: 0)
{'white': 104, 'asian': 6, 'black': 5, 'hawaiian': 3, 'other': 2}
  whitespace issues: 0 | empty strings: 0 | case variants: False

ethnicity  (missing: 0)
{'nonhispanic': 105, 'hispanic': 15}
  whitespace issues: 0 | empty strings: 0 | case variants: False

prefix  (missing: 26)
{'Mr.': 43, 'Mrs.': 37, nan: 26, 'Ms.': 14}
  whitespace issues: 0 | empty strings: 0 | case variants: False

suffix  (missing: 119)
{nan: 119, 'JD': 1}
  whitespace issues: 0 | empty strings: 0 | case variants: False
gender   F   M
prefix        
Mr.      0  43
Mrs.    37   0
Ms.     14   0
NA      12  14
marital
False    61.237509
True     15.813826
Name: birthdate, dtype: float64


### Patients: demographic columns

`gender`, `marital`, `race`, `ethnicity`, `prefix`, `suffix` all contain only valid categories, with no whitespace, case variants or empty strings. Prefix matches gender in all rows.

Missing `marital` (39) and `prefix` (26) are mostly minors (median age ~16 and ~12), so they are left as NaN. `suffix` is almost empty (1 value: "JD"). **No changes made.**

In [23]:
import pandas as pd
import numpy as np

# Reload ZIP and FIPS as strings. pandas reads ZIP as int, which drops leading zeros
# (01906 becomes 1906) and turns the '00000' placeholder into 0.
geo = pd.read_csv(base_url + "patients.csv", dtype={'ZIP': str, 'FIPS': str})
geo.columns = [c.lower() for c in geo.columns]
p = geo

MA_LAT, MA_LON = (41.2, 42.9), (-73.6, -69.9)   # Massachusetts bounding box
MA_COUNTY_FIPS = {'Barnstable':25001,'Berkshire':25003,'Bristol':25005,'Dukes':25007,
    'Essex':25009,'Franklin':25011,'Hampden':25013,'Hampshire':25015,'Middlesex':25017,
    'Nantucket':25019,'Norfolk':25021,'Plymouth':25023,'Suffolk':25025,'Worcester':25027}

def show(title, mask, cols):
    print(f"\n{title}: {int(mask.sum())}")
    if mask.sum():
        display(p.loc[mask, cols])

# 1. Missing values
print("Missing:\n", p[['state','county','city','zip','fips','lat','lon']].isna().sum())

# 2. ZIP format and range
p['zip_s'] = p['zip'].str.strip()
show("ZIP not numeric", ~p['zip_s'].str.fullmatch(r'\d+', na=False), ['id','city','zip'])
p['zip5'] = p['zip_s'].str.zfill(5)
show("ZIP placeholder 00000", p['zip5'] == '00000', ['city','county','zip5'])
valid_zip = p['zip5'] != '00000'
show("ZIP outside MA range (01001-02791)",
     valid_zip & ~p['zip5'].between('01001', '02791'), ['city','zip5'])

# 3. FIPS format
p['fips_s'] = p['fips'].str.replace(r'\.0$', '', regex=True)
show("FIPS not 5 digits", p['fips_s'].notna() & ~p['fips_s'].str.fullmatch(r'\d{5}', na=False),
     ['city','county','fips'])
show("FIPS not in Massachusetts (should start with 25)",
     p['fips_s'].notna() & ~p['fips_s'].str.startswith('25', na=False),
     ['city','county','state','fips_s'])

# 4. County vs FIPS consistency
p['county_name'] = p['county'].str.replace(' County', '', regex=False).str.strip()
p['expected_fips'] = p['county_name'].map(MA_COUNTY_FIPS).astype('Int64').astype(str)
show("FIPS disagrees with county name",
     p['fips_s'].notna() & (p['fips_s'] != p['expected_fips']),
     ['city','county','fips_s','expected_fips'])
print("\nMissing FIPS by county:\n", p[p['fips_s'].isna()]['county'].value_counts())

# 5. State / county / city / ZIP hierarchy
print("\nStates:", p['state'].value_counts().to_dict())
show("County not a MA county", ~p['county_name'].isin(MA_COUNTY_FIPS), ['city','county'])
multi = p.groupby('city')['county'].nunique()
show("City appears in >1 county", p['city'].isin(multi[multi > 1].index), ['city','county'])
multi_z = p[valid_zip].groupby('zip5')['city'].nunique()
show("ZIP maps to >1 city", p['zip5'].isin(multi_z[multi_z > 1].index), ['zip5','city','county'])

# 6. Coordinates
show("Missing lat/lon", p[['lat','lon']].isna().any(axis=1), ['city','lat','lon'])
show("Lat outside [-90, 90]", ~p['lat'].between(-90, 90) & p['lat'].notna(), ['city','lat'])
show("Lon outside [-180, 180]", ~p['lon'].between(-180, 180) & p['lon'].notna(), ['city','lon'])
show("(0,0) or swapped-looking coordinates",
     ((p['lat']==0)&(p['lon']==0)) | (p['lat'].abs() > 60) & (p['lon'].abs() < 60),
     ['city','lat','lon'])
show("Coordinates outside Massachusetts bounding box",
     ~p['lat'].between(*MA_LAT) | ~p['lon'].between(*MA_LON),
     ['city','county','lat','lon'])

# 7. Coordinates vs county (distance from each county's median point)
med = p.groupby('county')[['lat','lon']].transform('median')
p['km_from_county_median'] = np.hypot((p['lat']-med['lat'])*111, (p['lon']-med['lon'])*82)
show("Coordinates >60 km from their county's median point",
     p['km_from_county_median'] > 60, ['city','county','lat','lon','km_from_county_median'])

Missing:
 state      0
county     0
city       0
zip        0
fips      30
lat        0
lon        0
dtype: int64

ZIP not numeric: 0

ZIP placeholder 00000: 30


,city,county,zip5
12,Middleborough,Plymouth County,00000
16,Plympton,Plymouth County,00000
19,Wendell,Franklin County,00000
21,Northbridge,Worcester County,00000
25,Halifax,Plymouth County,00000
28,Plympton,Plymouth County,00000
34,Sterling,Worcester County,00000
35,Plympton,Plymouth County,00000
36,Mansfield,Bristol County,00000
38,Norton,Bristol County,00000



ZIP outside MA range (01001-02791): 1


,city,zip5
75,Attleboro,02861



FIPS not 5 digits: 0

FIPS not in Massachusetts (should start with 25): 1


...


FIPS disagrees with county name: 1


,city,county,fips_s,expected_fips
75,Attleboro,Bristol County,44007,25005



Missing FIPS by county:
 county
Plymouth County     6
Worcester County    6
Bristol County      5
Middlesex County    5
Norfolk County      3
Hampshire County    2
Franklin County     1
Berkshire County    1
Hampden County      1
Name: count, dtype: int64

States: {'Massachusetts': 120}

County not a MA county: 0

City appears in >1 county: 0

ZIP maps to >1 city: 2


,zip5,city,county
15,01906,Lynn,Essex County
108,01906,Saugus,Essex County



Missing lat/lon: 0

Lat outside [-90, 90]: 0

Lon outside [-180, 180]: 0

(0,0) or swapped-looking coordinates: 0

Coordinates outside Massachusetts bounding box: 0

Coordinates >60 km from their county's median point: 0


In [24]:
MA_COUNTY_FIPS = {'Barnstable':'25001','Berkshire':'25003','Bristol':'25005','Dukes':'25007',
    'Essex':'25009','Franklin':'25011','Hampden':'25013','Hampshire':'25015','Middlesex':'25017',
    'Nantucket':'25019','Norfolk':'25021','Plymouth':'25023','Suffolk':'25025','Worcester':'25027'}

# `geo` is the string-typed reload from the geographic check
g = geo.set_index('id')
g['county_name'] = g['county'].str.replace(' County', '', regex=False).str.strip()

# ZIP: keep leading zeros, placeholder -> NULL
g['zip'] = g['zip'].str.zfill(5).replace('00000', np.nan)
# Attleboro row with an RI ZIP -> NULL
g.loc[(g['city'] == 'Attleboro') & (g['zip'] == '02861'), 'zip'] = np.nan
# FIPS: derived from county (fixes the 30 missing and the Attleboro 44007 row)
g['fips'] = g['county_name'].map(MA_COUNTY_FIPS)

pt = data['patients']
pt['zip']  = pt['id'].map(g['zip'])
pt['fips'] = pt['id'].map(g['fips'])

# Readable labels; missing values stay NULL
pt['marital'] = pt['marital'].replace({'M': 'married', 'S': 'single', 'D': 'divorced', 'W': 'widowed'})
pt['gender']  = pt['gender'].replace({'F': 'female', 'M': 'male'})

print(pt[['zip', 'fips', 'marital', 'gender']].isna().sum())

zip        31
fips        0
marital    39
gender      0
dtype: int64


In [25]:
print(pt['fips'].str.len().value_counts().to_dict(), pt['zip'].str.len().value_counts().to_dict())
print(pt['fips'].dtype, pt['zip'].dtype)

{5: 120} {5.0: 89}
object object


### Patients: geographic fixes and encoding

- ZIP placeholder `00000` (30 rows) and the Attleboro row's RI ZIP `02861` set to NULL.
- FIPS derived from county name: fills 30 missing and fixes the Attleboro row (`44007` → `25005`).
- `zip` and `fips` kept as text to preserve leading zeros.
- `marital` and `gender` codes mapped to readable labels (e.g. `M` → `married`). Missing values stay NULL (mostly minors).

In [26]:
fin = ['healthcare_expenses', 'healthcare_coverage', 'income']
pt = data['patients']

# Age at death, or today for living patients (used to judge whether values are plausible)
end = pt['deathdate'].fillna(pd.Timestamp('2026-09-30'))
pt['age'] = (end - pt['birthdate']).dt.days / 365.25

# 1. Missing, negative, zero
print("Missing:", pt[fin].isna().sum().to_dict())
for c in fin:
    print(f"{c}: negative={(pt[c] < 0).sum()}, zero={(pt[c] == 0).sum()}")
print(pt[fin].describe().T)

# 2. Outliers: IQR x3 fence and log z-score, reported per column (nothing is removed)
flags = pd.DataFrame(index=pt.index)
for c in fin:
    q1, q3 = pt[c].quantile([.25, .75])
    fence = q3 + 3 * (q3 - q1)
    logz = np.log1p(pt[c])
    logz = (logz - logz.mean()) / logz.std()
    flags[f'{c}_iqr'] = pt[c] > fence
    flags[f'{c}_logz'] = logz.abs() > 3
    print(f"{c}: fence={fence:,.0f}, above fence={flags[f'{c}_iqr'].sum()}, "
          f"log-z>3={flags[f'{c}_logz'].sum()}, max={pt[c].max():,.0f}")

# 3. Context for judging legitimacy
print(pt[fin + ['age']].corr(method='spearman').round(2))

pt['expenses_per_year'] = pt['healthcare_expenses'] / pt['age'].clip(lower=1)
cols = ['age', 'deathdate'] + fin + ['expenses_per_year']
print("\nTop expenses:");  display(pt.nlargest(8, 'healthcare_expenses')[cols])
print("\nTop income:");    display(pt.nlargest(6, 'income')[cols])
print("\nTop coverage:");  display(pt.nlargest(6, 'healthcare_coverage')[cols])
print("\nLowest income:"); display(pt.nsmallest(5, 'income')[cols])
print("\nZero coverage:"); display(pt[pt['healthcare_coverage'] == 0][cols])

# 4. Suspicious repeated values
print(pt['income'].value_counts().head(5))

Missing: {'healthcare_expenses': 0, 'healthcare_coverage': 0, 'income': 0}
healthcare_expenses: negative=0, zero=0
healthcare_coverage: negative=0, zero=9
income: negative=0, zero=0
                     count           mean            std    min         25%  \
healthcare_expenses  120.0  152052.529083  191023.194260  400.0  24538.6275   
healthcare_coverage  120.0  348244.126500  431150.652181    0.0  18157.1625   
income               120.0  120476.175000  174472.938303   71.0  33128.7500   

                            50%          75%         max  
healthcare_expenses   89918.065  204184.6900   991285.28  
healthcare_coverage  165337.015  584063.1675  1946173.69  
income                79951.000  124338.2500   981530.00  
healthcare_expenses: fence=743,123, above fence=5, log-z>3=1, max=991,285
healthcare_coverage: fence=2,281,781, above fence=0, log-z>3=0, max=1,946,174
income: fence=397,967, above fence=6, log-z>3=1, max=981,530
                     healthcare_expenses  healthcare

,age,deathdate,healthcare_expenses,healthcare_coverage,income,expenses_per_year
90,61.095140,NaT,991285.28,573630.94,47167,16225.272172
53,62.625599,NaT,938185.96,439743.12,102442,14980.870066
29,71.364819,NaT,856338.00,629643.30,65031,11999.441974
110,81.664613,NaT,821047.68,342159.19,94798,10053.897852
117,64.427105,1986-02-23,773692.06,377262.92,79951,12008.797591
8,61.133470,NaT,591968.83,300378.23,95702,9683.219811
44,49.226557,NaT,506217.48,95602.97,37682,10283.422390
18,59.726215,NaT,487734.33,249038.25,82956,8166.168418



Top income:


,age,deathdate,healthcare_expenses,healthcare_coverage,income,expenses_per_year
27,42.951403,NaT,180000.31,538083.88,981530,4190.789981
103,48.566735,NaT,139764.65,650058.40,938744,2877.785581
108,46.992471,NaT,99565.36,31761.62,902392,2118.751325
13,52.164271,NaT,35849.94,162744.93,875139,687.250857
91,73.352498,NaT,164964.52,758008.33,634691,2248.928446
70,62.540726,NaT,186148.62,719021.13,472641,2976.438447



Top coverage:


...


Lowest income:


,age,deathdate,healthcare_expenses,healthcare_coverage,income,expenses_per_year
19,15.178645,NaT,24719.46,21918.32,71,1628.568320
89,3.411362,NaT,400.00,11107.77,2426,117.255217
39,67.457906,NaT,18474.70,974308.91,4412,273.870051
26,34.636550,NaT,11706.74,291852.88,4582,337.988047
12,64.016427,NaT,19115.40,1417903.14,7029,298.601482



Zero coverage:


,age,deathdate,healthcare_expenses,healthcare_coverage,income,expenses_per_year
5,5.486653,NaT,8837.38,0.0,50296,1610.705112
17,39.455168,NaT,135793.50,0.0,101170,3441.716458
23,6.340862,NaT,14571.95,0.0,53058,2298.102218
25,1.141684,NaT,3998.87,0.0,51090,3502.607356
38,19.195072,2026-06-18,110717.12,0.0,70638,5767.997159
40,1.018480,NaT,4926.41,0.0,98733,4837.019496
45,40.161533,2024-10-12,174083.26,0.0,76288,4334.577048
50,42.127310,NaT,109858.19,0.0,76288,2607.766550
102,11.613963,NaT,28641.53,0.0,47790,2466.128909


income
79951     8
24699     4
82756     3
82956     2
250398    2
Name: count, dtype: int64


### Patients: Checked for incomes, expences - nothing needs to be changed

In [27]:
PII_DROP = ['ssn', 'drivers', 'passport', 'address', 'maiden',
            'birthplace', 'prefix', 'suffix', 'middle']
helper_cols = ['age_at_reference', 'age', 'expenses_per_year']

pt = data['patients']
pt['lat'] = pt['lat'].round(2)   # ~1 km precision; use drop(columns=['lat','lon']) to remove entirely
pt['lon'] = pt['lon'].round(2)
data['patients'] = pt.drop(columns=PII_DROP + helper_cols, errors='ignore')

print(data['patients'].columns.tolist())

['id', 'birthdate', 'deathdate', 'first', 'last', 'marital', 'race', 'ethnicity', 'gender', 'city', 'state', 'county', 'fips', 'zip', 'lat', 'lon', 'healthcare_expenses', 'healthcare_coverage', 'income']


### Patients: Removed unnecessary for agent columns

In [28]:
import re

enc = data['encounters']

# 1. Non-null
null_count = enc['id'].isnull().sum()
print(f"Null IDs: {null_count}")

# 2. Whitespace (cleaned in place, as in the patients check)
ws_count = (enc['id'].astype(str) != enc['id'].astype(str).str.strip()).sum()
enc['id'] = enc['id'].astype(str).str.strip()
print(f"IDs with leading/trailing whitespace: {ws_count} (stripped)")

# 3. Unique
total_rows = len(enc)
unique_ids = enc['id'].nunique()
dup_count = enc['id'].duplicated().sum()
print(f"Unique IDs: {unique_ids} | Total rows: {total_rows} | Duplicated IDs: {dup_count}")
if unique_ids == total_rows:
    print("✓ All encounter IDs are unique.")
else:
    print("⚠️ WARNING: Duplicate encounter IDs detected!")
    display(enc[enc['id'].duplicated(keep=False)].sort_values('id'))

# 4. UUID format (8-4-4-4-12 hex, lowercase)
uuid_pattern = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$')
valid_mask = enc['id'].str.fullmatch(uuid_pattern.pattern)
print(f"Valid UUID format: {valid_mask.sum()} / {total_rows}")
if (~valid_mask).any():
    print(f"⚠️ WARNING: {(~valid_mask).sum()} IDs with invalid format:")
    print(enc.loc[~valid_mask, 'id'].head())
else:
    print("✓ All encounter IDs conform to UUID format.")

# 5. Lowercase check (mixed case would break joins in Postgres text comparisons)
print(f"IDs containing uppercase characters: {(enc['id'] != enc['id'].str.lower()).sum()}")

Null IDs: 0
IDs with leading/trailing whitespace: 0 (stripped)
Unique IDs: 7042 | Total rows: 7042 | Duplicated IDs: 0
✓ All encounter IDs are unique.
Valid UUID format: 7042 / 7042
✓ All encounter IDs conform to UUID format.
IDs containing uppercase characters: 0


### Encounters: Checked ids

In [29]:
pt  = data['patients']
enc = data['encounters']

# Make sure the keys are clean strings before comparing
enc['patient'] = enc['patient'].astype(str).str.strip()
pt['id']       = pt['id'].astype(str).str.strip()

# 1. encounters.patient → patients.id  (orphan encounters)
print(f"Null patient values in encounters: {enc['patient'].isna().sum()}")
orphan_ids = set(enc['patient']) - set(pt['id'])
orphan_rows = enc[enc['patient'].isin(orphan_ids)]
print(f"Patient IDs in encounters but not in patients: {len(orphan_ids)} "
      f"({len(orphan_rows)} encounter rows)")
if orphan_ids:
    print("⚠️ WARNING: orphan encounters detected!")
    display(orphan_rows.head())
else:
    print("✓ Every encounter belongs to a known patient.")

# 2. Reverse: patients with no encounters
enc_counts = enc.groupby('patient').size()
no_enc = pt[~pt['id'].isin(enc_counts.index)]
print(f"\nPatients with zero encounters: {len(no_enc)} / {len(pt)}")
if len(no_enc):
    display(no_enc[['id', 'birthdate', 'deathdate']])
else:
    print("✓ Every patient has at least one encounter.")

# 3. Distribution of encounters per patient
per_patient = pt['id'].map(enc_counts).fillna(0).astype(int)
print("\nEncounters per patient:")
print(per_patient.describe().round(1))
print(per_patient.sort_values(ascending=False).head(5).tolist())

Null patient values in encounters: 0
Patient IDs in encounters but not in patients: 0 (0 encounter rows)
✓ Every encounter belongs to a known patient.

Patients with zero encounters: 0 / 120
✓ Every patient has at least one encounter.

Encounters per patient:
count    120.0
mean      58.7
std       90.3
min        7.0
25%       23.0
50%       36.0
75%       54.0
max      584.0
Name: id, dtype: float64
[584, 536, 425, 404, 290]


### Encounters: Checked relations between patients and encounters - all clean

In [30]:
enc = data['encounters']

# 1. Parse (keep UTC, timezone-aware, as in the timeline checks)
enc['start_dt'] = pd.to_datetime(enc['start'], utc=True, errors='coerce')
enc['stop_dt']  = pd.to_datetime(enc['stop'],  utc=True, errors='coerce')
print("Unparsed start:", enc['start_dt'].isna().sum(), "| raw null:", enc['start'].isna().sum())
print("Unparsed stop: ", enc['stop_dt'].isna().sum(),  "| raw null:", enc['stop'].isna().sum())

# 2. Duration
enc['duration'] = enc['stop_dt'] - enc['start_dt']
enc['duration_hours'] = enc['duration'].dt.total_seconds() / 3600

print(f"\nNegative duration (stop < start): {(enc['duration'] < pd.Timedelta(0)).sum()}")
print(f"Zero duration:                    {(enc['duration'] == pd.Timedelta(0)).sum()}")
print(f"Stop in the future:               {(enc['stop_dt'] > pd.Timestamp.now(tz='UTC')).sum()}")

# 3. Overall summary
print("\nDuration: min", enc['duration'].min(),
      "| median", enc['duration'].median(), "| max", enc['duration'].max())
print(enc['duration'].quantile([.5, .9, .95, .99, .999]))

# 4. Per encounter class (a hospice stay and a check-up have very different normal lengths)
print(enc.groupby('encounterclass')['duration_hours'].describe().round(2))

# 5. Count of encounters over thresholds
for label, hrs in [('>24h', 24), ('>3 days', 72), ('>1 week', 168), ('>30 days', 720), ('>1 year', 8760)]:
    print(f"{label}: {(enc['duration_hours'] > hrs).sum()}")

# 6. Longest overall
cols = ['encounterclass', 'description', 'start', 'stop', 'duration_hours']
display(enc.nlargest(12, 'duration_hours')[cols])

# 7. Suspicious: short-visit classes lasting over 24h
short_classes = ['ambulatory', 'emergency', 'outpatient', 'wellness', 'urgentcare', 'virtual', 'home']
suspicious = enc[enc['encounterclass'].isin(short_classes) & (enc['duration_hours'] > 24)]
print(f"\nShort-visit classes lasting >24h: {len(suspicious)}")
display(suspicious[cols + ['reasondescription']].sort_values('duration_hours', ascending=False))

Unparsed start: 0 | raw null: 0
Unparsed stop:  0 | raw null: 0

Negative duration (stop < start): 0
Zero duration:                    0
Stop in the future:               0

Duration: min 0 days 00:15:00 | median 0 days 00:44:21 | max 100 days 00:15:00
0.500             0 days 00:44:21
0.900      0 days 03:33:56.600000
0.950   0 days 04:02:13.449999999
0.990      2 days 00:11:15.320000
0.999            30 days 00:15:00
Name: duration, dtype: timedelta64[ns]
                 count    mean     std    min     25%     50%     75%      max
encounterclass                                                                
ambulatory      3481.0    1.98    1.95   0.25    0.25    2.02    3.26    46.62
emergency        275.0    3.68   28.06   1.00    1.00    1.00    1.00   377.50
home              28.0    0.25    0.00   0.25    0.25    0.25    0.25     0.25
hospice           19.0  504.86  287.14  24.25  341.12  577.24  674.79  1152.25
inpatient         76.0   70.35   77.19  24.00   24.00   36.88   

,encounterclass,description,start,stop,duration_hours
3831,snf,Hospital admission (procedure),2023-06-02T06:27:29Z,2023-09-10T06:42:29Z,2400.250000
3268,snf,Hospital admission (procedure),2023-12-15T08:22:48Z,2024-02-22T08:37:48Z,1656.250000
1013,snf,Hospital admission (procedure),2020-09-05T01:30:03Z,2020-10-31T01:45:03Z,1344.250000
3119,snf,Hospital admission (procedure),2018-12-07T08:22:48Z,2019-01-29T08:37:48Z,1272.250000
5082,hospice,Admission to hospice (procedure),1996-09-24T22:08:49Z,1996-11-11T22:23:49Z,1152.250000
2138,wellness,Encounter for check up (procedure),2021-06-06T21:02:34Z,2021-07-21T09:03:42Z,1068.018889
1230,snf,Hospital admission (procedure),2024-06-25T02:42:37Z,2024-07-27T02:57:37Z,768.250000
794,hospice,Admission to hospice (procedure),2025-10-12T15:30:00Z,2025-11-11T15:45:00Z,720.250000
4493,snf,Hospital admission (procedure),2022-05-16T01:02:19Z,2022-06-15T01:17:19Z,720.250000
2659,hospice,Admission to hospice (procedure),2021-08-15T09:37:15Z,2021-09-13T09:52:15Z,696.250000



Short-visit classes lasting >24h: 6


,encounterclass,description,start,stop,duration_hours,reasondescription
2138,wellness,Encounter for check up (procedure),2021-06-06T21:02:34Z,2021-07-21T09:03:42Z,1068.018889,NaN
1789,emergency,Encounter for problem (procedure),1975-01-10T23:57:26Z,1975-01-26T17:27:26Z,377.500000,Sepsis (disorder)
2164,emergency,Encounter for problem (procedure),1992-04-10T20:07:56Z,1992-04-22T06:37:56Z,274.500000,Sepsis (disorder)
2140,outpatient,Patient encounter procedure (procedure),2021-07-21T09:39:01Z,2021-07-24T09:54:17Z,72.254444,NaN
473,ambulatory,Encounter for symptom (procedure),2006-07-12T00:53:42Z,2006-07-13T23:30:56Z,46.620556,Malignant neoplasm of breast (disorder)
715,ambulatory,Encounter for symptom (procedure),2015-08-28T15:30:00Z,2015-08-30T04:46:58Z,37.282778,Malignant neoplasm of breast (disorder)


In [31]:
enc['long_visit_flag'] = enc['encounterclass'].isin(short_classes) & (enc['duration_hours'] > 24)

### Encounters: dates and duration

`start` and `stop` parse for all 7,042 rows; no negative or zero durations. Median 44 min, max 100 days. Long stays are in snf, hospice and inpatient classes (expected).

6 short-visit encounters exceed 24h (wellness 44 days, 2 emergency sepsis 11-16 days, 1 outpatient 3 days, 2 ambulatory ~1.5 days); kept and flagged with `long_visit_flag`. **No rows removed.**

In [32]:
enc = data['encounters']
pt  = data['patients']

# Parse to tz-naive UTC (birth/death are date-only, so tolerance handles the mismatch)
enc['start_dt'] = pd.to_datetime(enc['start'], utc=True).dt.tz_localize(None)
enc['stop_dt']  = pd.to_datetime(enc['stop'],  utc=True).dt.tz_localize(None)

chk = enc.merge(
    pt[['id', 'birthdate', 'deathdate']].rename(columns={'id': 'pid'}),
    left_on='patient', right_on='pid', how='left'
)
print("Encounters with no matching patient:", chk['birthdate'].isna().sum())

tol = pd.Timedelta(days=1)
now = pd.Timestamp.now(tz='UTC').tz_localize(None)

# 1. Overall time range
print("Encounter start range:", chk['start_dt'].min(), "→", chk['start_dt'].max())
print("Latest stop:          ", chk['stop_dt'].max())

# 2. Future encounters
print("\nStart in the future:", (chk['start_dt'] > now).sum())
print("Stop in the future: ", (chk['stop_dt']  > now).sum())

# 3. Before birth: strict vs 1-day tolerance
print("\nBefore birth (strict):        ", (chk['start_dt'] < chk['birthdate']).sum())
print("Before birth (1-day tolerance):", (chk['start_dt'] < chk['birthdate'] - tol).sum())

# 4. After death: strict vs tolerance, using start and stop
dead = chk['deathdate'].notna()
print("\nStart after death (strict):        ", (dead & (chk['start_dt'] > chk['deathdate'])).sum())
print("Start after death (1-day tolerance):", (dead & (chk['start_dt'] > chk['deathdate'] + tol)).sum())
after_death = chk[dead & (chk['stop_dt'] > chk['deathdate'] + tol)]
print("Stop after death (1-day tolerance): ", len(after_death))
print(after_death['description'].value_counts())

# Encounters that begin before death and run past it (clinical care that overlaps death)
straddle = after_death[after_death['start_dt'] <= after_death['deathdate'] + tol]
print("Started before death but ended after it:", len(straddle))

# 5. Per-patient sanity: last encounter vs death, first encounter vs birth
last = chk[dead].groupby('patient').agg(last_start=('start_dt', 'max'), death=('deathdate', 'first'))
print("\nDays from death to last encounter (deceased patients):")
print((last['last_start'] - last['death']).dt.days.describe())
first = chk.groupby('patient').agg(first=('start_dt', 'min'), birth=('birthdate', 'first'))
print("Days from birth to first encounter:", (first['first'] - first['birth']).dt.days.min(), "min")

Encounters with no matching patient: 0
Encounter start range: 1938-05-30 18:52:05 → 2026-09-23 09:24:06
Latest stop:           2026-09-23 10:05:08

Start in the future: 0
Stop in the future:  0

Before birth (strict):         4
Before birth (1-day tolerance): 0

Start after death (strict):         21
Start after death (1-day tolerance): 18
Stop after death (1-day tolerance):  18
description
Death Certification    18
Name: count, dtype: int64
Started before death but ended after it: 0

Days from death to last encounter (deceased patients):
count    20.000000
mean      5.650000
std       4.221187
min       0.000000
25%       2.000000
50%       5.000000
75%       9.000000
max      14.000000
dtype: float64
Days from birth to first encounter: -1 min


### Encounters: cross-table date validation

Range 1938-05-30 → 2026-09-23; **0 future encounters**. With a 1-day tolerance, 0 encounters before birth and 18 after death (all "Death Certification"); none overlap death. Each of the 20 deceased patients' last encounter falls 0-14 days after the death date. **No rows removed** (flag `is_death_certification` already added).

In [33]:
enc = data['encounters']
col = enc['encounterclass']

expected = {'wellness', 'ambulatory', 'urgentcare', 'inpatient', 'outpatient',
            'emergency', 'snf', 'hospice', 'virtual', 'home'}

# 1. Missing and dtype
print("Missing:", col.isna().sum(), "| dtype:", col.dtype)

# 2. Category counts
print(col.value_counts(dropna=False))

# 3. Unexpected categories (compared to your expected list)
found = set(col.dropna().unique())
print("\nUnexpected categories:", found - expected or "none")
print("Expected but absent:  ", expected - found or "none")

# 4. Whitespace, blanks, case
v = col.dropna().astype(str)
print("\nWhitespace issues:", (v != v.str.strip()).sum())
print("Empty strings:    ", (v.str.strip() == '').sum())
print("Not lowercase:    ", (v != v.str.lower()).sum())
print("Only a-z letters: ", v.str.fullmatch(r'[a-z]+').all())

# 5. Spelling variants: distinct values before vs after normalizing
print(f"Distinct raw: {v.nunique()} | after strip+lower: {v.str.strip().str.lower().nunique()}")

# 6. Relationship to description/code (does description determine the class?)
print("\nMax classes per code:       ", enc.groupby('code')['encounterclass'].nunique().max())
print("Max descriptions per code:  ", enc.groupby('code')['description'].nunique().max())
print(enc.groupby('description')['encounterclass'].nunique().sort_values(ascending=False).head(5))

Missing: 0 | dtype: object
encounterclass
ambulatory    3481
wellness      1521
outpatient    1027
urgentcare     584
emergency      275
inpatient       76
home            28
hospice         19
snf             16
virtual         15
Name: count, dtype: int64

Unexpected categories: none
Expected but absent:   none

Whitespace issues: 0
Empty strings:     0
Not lowercase:     0
Only a-z letters:  True
Distinct raw: 10 | after strip+lower: 10

Max classes per code:        5
Max descriptions per code:   1
description
Encounter for problem (procedure)          5
Encounter for check up (procedure)         4
Hospital admission (procedure)             3
Patient encounter procedure (procedure)    3
Follow-up encounter (procedure)            2
Name: encounterclass, dtype: int64


### Encounters: encounter class

10 categories exactly as expected (ambulatory 3,481; wellness 1,521; outpatient 1,027; urgentcare 584; emergency 275; inpatient 76; home 28; hospice 19; snf 16; virtual 15). No missing values, whitespace, case or spelling variants. Class is independent of code/description (one code appears under up to 5 classes). **No changes made.**

In [34]:
enc = data['encounters']
cost_cols = ['base_encounter_cost', 'total_claim_cost', 'payer_coverage']
tol = 0.01   # ignore sub-cent rounding

# 1. Missing / negative / zero
print("Missing:", enc[cost_cols].isna().sum().to_dict())
for c in cost_cols:
    print(f"{c}: negative={(enc[c] < 0).sum()}, zero={(enc[c] == 0).sum()}")
print(enc[cost_cols].describe().T.round(2))

# 2. Coverage cannot exceed the total claim
cov_gt_total = enc[enc['payer_coverage'] > enc['total_claim_cost'] + tol]
print(f"\npayer_coverage > total_claim_cost: {len(cov_gt_total)}")
if len(cov_gt_total):
    display(cov_gt_total[['encounterclass', 'description'] + cost_cols].head(10))

# 3. Total claim below the base encounter cost
total_lt_base = enc[enc['total_claim_cost'] < enc['base_encounter_cost'] - tol]
print(f"total_claim_cost < base_encounter_cost: {len(total_lt_base)}")
if len(total_lt_base):
    display(total_lt_base[['encounterclass', 'description'] + cost_cols].head(10))

# 4. Relationships worth knowing
enc['patient_pays'] = enc['total_claim_cost'] - enc['payer_coverage']
print("\nTotal == base cost:          ", np.isclose(enc['total_claim_cost'], enc['base_encounter_cost']).sum())
print("Coverage == total (100% paid):", (np.isclose(enc['payer_coverage'], enc['total_claim_cost']) & (enc['total_claim_cost'] > 0)).sum())
print("Patient share (total - coverage):")
print(enc['patient_pays'].describe().round(2))

# 5. Zero coverage: by class and by payer
zero = enc['payer_coverage'] == 0
print("\nZero-coverage encounters:", zero.sum(), f"({zero.mean():.1%})")
print(pd.crosstab(enc['encounterclass'], zero))
by_payer = zero.groupby(enc['payer']).agg(['sum', 'size'])
by_payer['share'] = (by_payer['sum'] / by_payer['size']).round(2)
print(by_payer.sort_values('size', ascending=False).head(8))

# 6. Largest claims (context, not removal)
display(enc.nlargest(6, 'total_claim_cost')[['encounterclass', 'description'] + cost_cols])
print("Cost columns with more than 2 decimals:",
      {c: int((enc[c].round(2) != enc[c]).sum()) for c in cost_cols})

Missing: {'base_encounter_cost': 0, 'total_claim_cost': 0, 'payer_coverage': 0}
base_encounter_cost: negative=0, zero=0
total_claim_cost: negative=0, zero=0
payer_coverage: negative=0, zero=1919
                      count     mean      std   min     25%     50%      75%  \
base_encounter_cost  7042.0   112.15    27.38  75.0   85.55   85.55   142.58   
total_claim_cost     7042.0  2103.24  5212.73  75.0  516.95  872.30  1791.03   
payer_coverage       7042.0  1404.33  4167.41   0.0    0.00  485.87  1131.38   

                           max  
base_encounter_cost     146.18  
total_claim_cost     262086.81  
payer_coverage       209669.21  

payer_coverage > total_claim_cost: 0
total_claim_cost < base_encounter_cost: 0

Total == base cost:           725
Coverage == total (100% paid): 532
Patient share (total - coverage):
count     7042.00
mean       698.91
std       1918.05
min          0.00
25%         50.00
50%        174.48
75%        707.35
max      54492.94
Name: patient_pays, dtyp

,encounterclass,description,base_encounter_cost,total_claim_cost,payer_coverage
4795,inpatient,Admission to intensive care unit (procedure),146.18,262086.81,209669.21
3655,ambulatory,Prenatal initial visit (regime/therapy),142.58,71529.19,57223.35
3831,snf,Hospital admission (procedure),110.92,69347.33,69347.33
2164,emergency,Encounter for problem (procedure),87.71,55667.07,29899.77
1662,ambulatory,Prenatal initial visit (regime/therapy),142.58,54492.94,0.00
1323,ambulatory,Prenatal initial visit (regime/therapy),142.58,50148.78,38952.63


Cost columns with more than 2 decimals: {'base_encounter_cost': 0, 'total_claim_cost': 0, 'payer_coverage': 0}


### Encounters: financial logic

No missing or negative costs. `payer_coverage <= total_claim_cost` holds in all 7,042 rows, and `total_claim_cost >= base_encounter_cost` in all rows (base cost appears to be only the encounter fee; total also includes billed procedures/medications).

1,919 encounters (27%) have zero coverage: mostly wellness (53% of wellness visits), plus one payer with 100% zero coverage (likely "no insurance", unconfirmed). Kept as valid. Largest claim $262K (inpatient ICU). **No changes made.**

In [35]:
enc = data['encounters']
rc_null = enc['reasoncode'].isna()
rd_null = enc['reasondescription'].isna()

# 1. Do the two columns go missing together?
print(f"reasoncode null: {rc_null.sum()} ({rc_null.mean():.1%}) | reasondescription null: {rd_null.sum()}")
print(pd.crosstab(rc_null.rename('code_null'), rd_null.rename('desc_null')))
print("Code missing, description present:", (rc_null & ~rd_null).sum())
print("Code present, description missing:", (~rc_null & rd_null).sum())

# 2. Blank strings / whitespace pretending to be values
v = enc['reasondescription'].dropna()
print("\nEmpty strings:", (v.str.strip() == '').sum(), "| whitespace issues:", (v != v.str.strip()).sum())

# 3. Is it a 1:1 code <-> description mapping?
has = enc.dropna(subset=['reasoncode'])
print("Descriptions per code (max):", has.groupby('reasoncode')['reasondescription'].nunique().max())
print("Codes per description (max):", has.groupby('reasondescription')['reasoncode'].nunique().max())
print("Distinct reason codes:", enc['reasoncode'].nunique())
print("All codes whole numbers:", (has['reasoncode'] == has['reasoncode'].astype(int)).all())

# 4. Does missingness follow the type of encounter?
print("\nShare with NO reason, by class:")
print(rc_null.groupby(enc['encounterclass']).agg(['sum', 'size', 'mean']).round(2)
      .sort_values('mean', ascending=False))
print("\nMost common encounter descriptions and their null share:")
print(enc.groupby('description').agg(n=('id', 'size'), null_share=('reasoncode', lambda s: s.isna().mean()))
      .sort_values('n', ascending=False).head(12).round(2))

# 5. Top reasons
print(enc['reasondescription'].value_counts().head(10))

reasoncode null: 3209 (45.6%) | reasondescription null: 3209
desc_null  False  True 
code_null              
False       3833      0
True           0   3209
Code missing, description present: 0
Code present, description missing: 0

Empty strings: 0 | whitespace issues: 0
Descriptions per code (max): 1
Codes per description (max): 1
Distinct reason codes: 108
All codes whole numbers: True

Share with NO reason, by class:
                 sum  size  mean
encounterclass                  
urgentcare       582   584  1.00
wellness        1501  1521  0.99
outpatient       767  1027  0.75
emergency        119   275  0.43
hospice            6    19  0.32
ambulatory       233  3481  0.07
virtual            1    15  0.07
home               0    28  0.00
inpatient          0    76  0.00
snf                0    16  0.00

Most common encounter descriptions and their null share:
                                                       n  null_share
description                                          

In [36]:
enc['reasoncode'] = enc['reasoncode'].astype('Int64')   # nullable; becomes bigint in Postgres

### Encounters: reason fields

`reasoncode` and `reasondescription` are missing together in exactly the same 3,209 rows (45.6%); no partial records, no blanks, strict 1:1 code↔description mapping (108 codes). Missingness follows visit type: urgentcare 99.7%, wellness 98.7%, outpatient 75% null vs inpatient/snf/home 0%, so NULL means "no specific reason recorded", not a data error.

**Kept as NULL** (no "Unknown" fill); `reasoncode` cast to nullable `Int64`.

In [37]:
con = data['conditions']
pt  = data['patients']
enc = data['encounters']

# 1. Parse (conditions dates are date-only strings)
con['start_d'] = pd.to_datetime(con['start'], errors='coerce')
con['stop_d']  = pd.to_datetime(con['stop'],  errors='coerce')
print("Rows:", len(con), "| missing stop:", con['stop'].isna().sum(), f"({con['stop'].isna().mean():.1%})")
print("Unparsed start:", con['start_d'].isna().sum(),
      "| unparsed stop (non-null raw):", (con['stop_d'].isna() & con['stop'].notna()).sum())

# 2. stop vs start (only where stop exists)
print("\nstop < start:", (con['stop_d'] < con['start_d']).sum())
print("stop == start (same-day):", (con['stop_d'] == con['start_d']).sum())
dur = (con['stop_d'] - con['start_d']).dt.days
print(dur.describe().round(1))
display(con.loc[dur.nlargest(5).index, ['description', 'start', 'stop']].assign(days=dur.nlargest(5)))

# 3. Birth / death (both are date-only here, so no tolerance is needed)
c = con.merge(pt[['id', 'birthdate', 'deathdate']].rename(columns={'id': 'pid'}),
              left_on='patient', right_on='pid', how='left')
dead = c['deathdate'].notna()
print("\nNo matching patient:", c['birthdate'].isna().sum())
print("start < birth:", (c['start_d'] < c['birthdate']).sum())
print("start > death:", (dead & (c['start_d'] > c['deathdate'])).sum())
print("stop  > death:", (dead & (c['stop_d']  > c['deathdate'])).sum())

# 4. Overlap with the linked encounter (encounters are UTC timestamps, so use a 1-day tolerance)
e = enc[['id', 'start', 'stop', 'patient', 'encounterclass', 'description']].copy()
e['e_start'] = pd.to_datetime(e['start'], utc=True).dt.tz_localize(None).dt.normalize()
e['e_stop']  = pd.to_datetime(e['stop'],  utc=True).dt.tz_localize(None).dt.normalize()
e = e.rename(columns={'id': 'eid', 'patient': 'e_patient', 'description': 'enc_description'})
m = c.merge(e[['eid', 'e_start', 'e_stop', 'e_patient', 'encounterclass', 'enc_description']],
            left_on='encounter', right_on='eid', how='left')

tol = pd.Timedelta(days=1)
print("\nNo matching encounter:", m['eid'].isna().sum())
print("Patient differs from encounter's patient:", (m['patient'] != m['e_patient']).sum())
print("Start before encounter (strict):", (m['start_d'] < m['e_start']).sum())
print("Start after encounter end (strict):", (m['start_d'] > m['e_stop']).sum())
print("Start after encounter end (1-day tolerance):", (m['start_d'] > m['e_stop'] + tol).sum())

late = m[m['start_d'] > m['e_stop'] + tol].copy()
late['days_after_encounter'] = (late['start_d'] - late['e_stop']).dt.days
display(late[['description', 'start', 'encounterclass', 'enc_description', 'days_after_encounter']]
        .sort_values('days_after_encounter', ascending=False))

z = m[m['stop_d'].notna()]
print("Resolved: stop before encounter start:", (z['stop_d'] < z['e_start']).sum(),
      "| after encounter end:", (z['stop_d'] > z['e_stop']).sum())

Rows: 5099 | missing stop: 1187 (23.3%)
Unparsed start: 0 | unparsed stop (non-null raw): 0

stop < start: 0
stop == start (same-day): 782
count     3912.0
mean       272.6
std        749.4
min          0.0
25%         10.0
50%         28.0
75%        266.0
max      14077.0
dtype: float64


,description,start,stop,days
538,Limited social contact (finding),1983-05-29,2021-12-12,14077.0
539,Social isolation (finding),1985-12-15,2019-12-01,12404.0
1670,Chronic kidney disease stage 1 (disorder),1988-05-20,2017-07-14,10647.0
2567,Social isolation (finding),1999-02-14,2026-06-14,9982.0
200,Full-time employment (finding),1985-03-03,2009-01-18,8722.0



No matching patient: 0
start < birth: 0
start > death: 0
stop  > death: 0

No matching encounter: 0
Patient differs from encounter's patient: 0
Start before encounter (strict): 0
Start after encounter end (strict): 1309
Start after encounter end (1-day tolerance): 29


,description,start,encounterclass,enc_description,days_after_encounter
2550,Ischemic heart disease (disorder),2022-07-06,outpatient,Administration of vaccine to produce active im...,344
2958,Suspected disease caused by Severe acute respi...,2021-01-06,ambulatory,Encounter for check up (procedure),324
1134,Suspected disease caused by Severe acute respi...,2020-12-31,wellness,Well child visit (procedure),242
1741,Pulmonary emphysema (disorder),1996-08-08,outpatient,Consultation for treatment (procedure),223
2599,Acute infective cystitis (disorder),2022-12-25,wellness,General examination of patient (procedure),217
3707,Suspected disease caused by Severe acute respi...,2020-12-08,wellness,General examination of patient (procedure),211
1451,Suspected disease caused by Severe acute respi...,2021-01-21,ambulatory,Ophthalmic examination and evaluation (procedure),201
2708,Acute infective cystitis (disorder),1985-03-28,emergency,Emergency room admission (procedure),165
1381,Suspected disease caused by Severe acute respi...,2021-02-19,wellness,Well child visit (procedure),112
412,Suspected disease caused by Severe acute respi...,2020-11-13,ambulatory,Encounter for check up (procedure),88


Resolved: stop before encounter start: 0 | after encounter end: 3309


In [38]:
con['start_after_encounter_flag'] = con.index.isin(late.index)

### Conditions: date logic

`stop` missing in 1,187 rows (23.3%): ongoing/chronic, kept as NULL. No `stop < start`, no conditions before birth or after death, all rows link to a valid encounter of the same patient. 1,309 conditions start after their encounter (UTC/date-only mismatch); 29 remain with a 1-day tolerance, starting 3-344 days after the linked encounter (mostly suspected COVID-19 and cystitis).

**No rows removed**; the 29 are flagged with `start_after_encounter_flag`.

In [39]:
con = data['conditions']
pt  = data['patients']
enc = data['encounters']

# Clean keys (same as before)
for df, cols in [(pt, ['id']), (enc, ['id', 'patient']), (con, ['patient', 'encounter'])]:
    for c in cols:
        df[c] = df[c].astype(str).str.strip()

print("Null patient / encounter in conditions:",
      con['patient'].isna().sum(), "/", con['encounter'].isna().sum())

# 1. conditions.patient → patients.id
orph_p = set(con['patient']) - set(pt['id'])
print(f"\nOrphan patient IDs: {len(orph_p)} ({con['patient'].isin(orph_p).sum()} rows)")

# 2. conditions.encounter → encounters.id
orph_e = set(con['encounter']) - set(enc['id'])
print(f"Orphan encounter IDs: {len(orph_e)} ({con['encounter'].isin(orph_e).sum()} rows)")

# 3. Patient consistency: conditions.patient == encounters[conditions.encounter].patient
chk = con.merge(
    enc[['id', 'patient']].rename(columns={'id': 'eid', 'patient': 'enc_patient'}),
    left_on='encounter', right_on='eid', how='left'
)
print(f"\nRows after merge: {len(chk)} (should equal {len(con)}; more means duplicate encounter IDs)")
mismatch = chk[chk['patient'] != chk['enc_patient']]
print(f"Patient mismatches (condition vs its encounter): {len(mismatch)}")
if len(mismatch):
    display(mismatch[['patient', 'encounter', 'enc_patient', 'description']].head(10))
else:
    print("✓ Every condition's patient matches its encounter's patient.")

# 4. Reverse coverage (informational, not errors)
print(f"\nPatients with conditions: {con['patient'].nunique()} / {len(pt)}")
print(f"Encounters with conditions: {con['encounter'].nunique()} / {len(enc)}")
print("Share of encounters that have conditions, by class:")
print(enc.assign(has=enc['id'].isin(con['encounter'])).groupby('encounterclass')['has'].mean().round(2))
print("\nConditions per patient:"); print(con.groupby('patient').size().describe().round(1))
print("Max conditions on one encounter:", con.groupby('encounter').size().max())

Null patient / encounter in conditions: 0 / 0

Orphan patient IDs: 0 (0 rows)
Orphan encounter IDs: 0 (0 rows)

Rows after merge: 5099 (should equal 5099; more means duplicate encounter IDs)
Patient mismatches (condition vs its encounter): 0
✓ Every condition's patient matches its encounter's patient.

Patients with conditions: 120 / 120
Encounters with conditions: 3289 / 7042
Share of encounters that have conditions, by class:
encounterclass
ambulatory    0.23
emergency     0.74
home          0.00
hospice       0.00
inpatient     0.64
outpatient    0.53
snf           0.00
urgentcare    0.72
virtual       0.07
wellness      0.83
Name: has, dtype: float64

Conditions per patient:
count    120.0
mean      42.5
std       53.4
min        4.0
25%       20.8
50%       29.5
75%       38.2
max      303.0
dtype: float64
Max conditions on one encounter: 7


### Conditions: patient and encounter relationships

All conditions reference an existing patient and encounter (0 orphans). Every condition's patient matches its encounter's patient (0 mismatches; merge preserved all 5,099 rows). All 120 patients have conditions; 47% of encounters do (0% for home/hospice/snf, 23% ambulatory). **No changes made.**

In [40]:
# Read the raw column as text so leading zeros / long values can't be altered
raw = pd.read_csv(base_url + "conditions.csv", dtype=str)
raw.columns = [c.lower() for c in raw.columns]
con = data['conditions']

# 1. Missing
print("Null code / description / system:",
      con['code'].isna().sum(), "/", con['description'].isna().sum(), "/", con['system'].isna().sum())

# 2. Numeric, whitespace, leading zeros
s = raw['code']
print("\nNon-numeric codes:", (~s.str.fullmatch(r'\d+', na=False)).sum())
print("Whitespace issues:", (s != s.str.strip()).sum(), "| leading zeros:", s.str.startswith('0').sum())

# 3. Code lengths
print("\nCode length distribution:")
print(s.str.len().value_counts().sort_index())
raw['len'] = s.str.len()
display(raw[raw['len'].isin([7, 10, 13, 14, 15, 17])]
        .groupby(['len', 'code', 'description']).size().reset_index(name='rows'))

# 4. Code <-> description mapping
print("\nDistinct codes:", con['code'].nunique(),
      "| distinct descriptions:", con['description'].nunique(),
      "| distinct (code, description) pairs:", con[['code', 'description']].drop_duplicates().shape[0])
per_code = con.groupby('code')['description'].nunique()
per_desc = con.groupby('description')['code'].nunique()
print("Codes with >1 description:", (per_code > 1).sum())
print("Descriptions with >1 code:", (per_desc > 1).sum())
if (per_code > 1).any():
    display(con[con['code'].isin(per_code[per_code > 1].index)]
            .groupby(['code', 'description']).size().reset_index(name='rows'))

# 5. Description text quality
d = con['description']
print("\nDescription whitespace:", (d != d.str.strip()).sum(),
      "| blank:", (d.str.strip() == '').sum(),
      "| double spaces:", d.str.contains('  ').sum())
print("Distinct after lower+strip:", d.str.lower().str.strip().nunique())

# 6. System and duplicate rows
print("\nSystem values:", con['system'].value_counts().to_dict())
print("Exact duplicate rows:", con.duplicated().sum(),
      "| duplicates on (patient, encounter, code, start):",
      con.duplicated(['patient', 'encounter', 'code', 'start']).sum())

# 7. Precision risk: codes above the largest integer JavaScript can hold exactly (2**53)
big = con['code'].astype('int64') > 2**53
print("\nCodes > 2**53:", big.sum(), "rows,", con.loc[big, 'code'].nunique(), "distinct code(s)")

Null code / description / system: 0 / 0 / 0

Non-numeric codes: 0
Whitespace issues: 0 | leading zeros: 0

Code length distribution:
code
7        6
8     1466
9     3536
10      19
13      16
14      14
15      20
17      22
Name: count, dtype: int64


,len,code,description,rows
0,7,1734006,Fracture of vertebral column with spinal cord ...,1
1,7,4557003,Preinfarction syndrome (disorder),1
2,7,5602001,Opioid abuse,1
3,7,6525002,Dependent drug abuse (disorder),3
4,10,1149222004,Overdose (disorder),9
5,10,1187604002,Serving in military service (finding),4
6,10,1290882004,History of seizure (situation),6
7,13,1121000119107,Chronic neck pain (finding),11
8,13,1231000119100,History of aortic valve replacement (situation),1
9,13,1551000119108,Nonproliferative diabetic retinopathy due to t...,4



Distinct codes: 185 | distinct descriptions: 185 | distinct (code, description) pairs: 185
Codes with >1 description: 0
Descriptions with >1 code: 0

Description whitespace: 0 | blank: 0 | double spaces: 4
Distinct after lower+strip: 185

System values: {'http://snomed.info/sct': 5099}
Exact duplicate rows: 0 | duplicates on (patient, encounter, code, start): 0

Codes > 2**53: 22 rows, 1 distinct code(s)


In [41]:
data['conditions']['code'] = data['conditions']['code'].astype(str)

### Conditions: codes

`code`, `description`, `system` have no nulls; all codes are numeric, no whitespace. 185 codes ↔ 185 descriptions, a strict 1:1 mapping (no code with two meanings). No duplicate rows. Lengths 7-17 digits are valid SNOMED IDs (13-17 digits contain `1000119`, the US-extension pattern). One code exceeds 2^53, so `code` is stored as **text** to avoid precision loss in JSON clients.

Not all conditions are diseases (findings, situations, employment status). **No rows removed.**

In [42]:
con = data['conditions']
key = ['patient', 'encounter', 'code', 'start']

# 1. Exact and key-based duplicates
print("Exact duplicate rows:", con.duplicated().sum())
print("Duplicates on", key, ":", con.duplicated(key).sum(),
      "| rows involved:", con.duplicated(key, keep=False).sum())

# 2. Looser keys: rows that differ only in a column (e.g. stop) but describe the same event
loose_keys = {
    'patient+code+start':            ['patient', 'code', 'start'],
    'patient+encounter+code':        ['patient', 'encounter', 'code'],
    'encounter+code+start':          ['encounter', 'code', 'start'],
    'patient+encounter+start+desc':  ['patient', 'encounter', 'start', 'description'],
}
for name, k in loose_keys.items():
    print(f"{name}: {con.duplicated(k).sum()} duplicates")

# 3. Same condition repeated for a patient: separate episodes or overlapping duplicates?
c = con.copy()
c['s'] = pd.to_datetime(c['start']); c['e'] = pd.to_datetime(c['stop'])
c = c.sort_values(['patient', 'code', 's'])
g = c.groupby(['patient', 'code'])
c['prev_s'], c['prev_e'], c['n'] = g['s'].shift(), g['e'].shift(), g.cumcount()
rep = c[c['n'] > 0].copy()

print(f"\nRepeated patient+code rows: {len(rep)} (patients x conditions with >1 episode: {(g.size() > 1).sum()})")
print("Repeat while previous episode still open (no stop):", rep['prev_e'].isna().sum())
print("Repeat starting before previous episode ended:     ", (rep['s'] < rep['prev_e']).sum())
print("Repeat with the same start date:                   ", (rep['s'] == rep['prev_s']).sum())
gap = (rep['s'] - rep['prev_e']).dt.days
print("Gap from previous stop to next start (days):")
print(gap.describe().round(1))
print("Gap of 0 days:", (gap == 0).sum(), "| within 7 days:", (gap <= 7).sum())

# 4. Which conditions repeat most
print(rep['description'].value_counts().head(8))
print(g.size().sort_values(ascending=False).head(3))

Exact duplicate rows: 0
Duplicates on ['patient', 'encounter', 'code', 'start'] : 0 | rows involved: 0
patient+code+start: 0 duplicates
patient+encounter+code: 0 duplicates
encounter+code+start: 0 duplicates
patient+encounter+start+desc: 0 duplicates

Repeated patient+code rows: 2646 (patients x conditions with >1 episode: 651)
Repeat while previous episode still open (no stop): 0
Repeat starting before previous episode ended:      0
Repeat with the same start date:                    0
Gap from previous stop to next start (days):
count    2646.0
mean      362.2
std       519.2
min         0.0
25%        28.0
50%       154.0
75%       371.0
max      3325.0
dtype: float64
Gap of 0 days: 2 | within 7 days: 247
description
Medication review due (situation)    964
Stress (finding)                     335
Gingivitis (disorder)                311
Full-time employment (finding)       271
Part-time employment (finding)       159
Limited social contact (finding)      88
Not in labor force (find

### Conditions: duplicate clinical events

No duplicates on `patient + encounter + code + start`, nor on looser keys (patient+code+start, patient+encounter+code, encounter+code+start). 2,646 repeated patient+code rows are separate episodes: none overlaps the previous episode or repeats while one is still open (median gap 154 days); mostly recurring items like "Medication review due". **No rows removed.**

### Medications: date logic

`stop` missing in 348 rows (4.3%): ongoing, kept as NULL. All 8,001 rows link to a valid patient and encounter; none before birth or after death (6 ICU meds on the death date, within tolerance), none start before their encounter.

- **6 negative durations** (2 patients; stop 2-6 days before start): kept and flagged with `stop_before_start_flag`.
- **1,614 zero-duration** rows are one-time administrations (injections, gels, eye drops): valid.
- Longest durations are chronic statins (up to 47 years): plausible.
- 81 meds start 1-344 days after their encounter ended (mostly a 7-day follow-up): kept.

**No rows removed.**

In [43]:
med = data['medications']
pt  = data['patients']
enc = data['encounters']

# 1. Parse (full UTC timestamps, same as encounters)
med['start_dt'] = pd.to_datetime(med['start'], utc=True, errors='coerce').dt.tz_localize(None)
med['stop_dt']  = pd.to_datetime(med['stop'],  utc=True, errors='coerce').dt.tz_localize(None)
print("Rows:", len(med), "| missing stop:", med['stop'].isna().sum(), f"({med['stop'].isna().mean():.1%})")
print("Unparsed start:", med['start_dt'].isna().sum(),
      "| unparsed stop (non-null raw):", (med['stop_dt'].isna() & med['stop'].notna()).sum())

# 2. Duration
med['duration'] = med['stop_dt'] - med['start_dt']
med['duration_days'] = med['duration'].dt.total_seconds() / 86400
print("\nNegative duration:", (med['duration'] < pd.Timedelta(0)).sum())
print("Zero duration:    ", (med['duration'] == pd.Timedelta(0)).sum())
print(med['duration_days'].describe().round(2))
print(med['duration_days'].quantile([.5, .9, .95, .99, .999]).round(1))
for d in [1, 30, 365, 3650, 7300]:
    print(f"> {d} days: {(med['duration_days'] > d).sum()}")

cols = ['patient', 'encounter', 'description', 'start', 'stop', 'dispenses', 'totalcost']
print("\nNegative durations:")
display(med[med['duration'] < pd.Timedelta(0)][cols])
print("Zero-duration medications (top):")
print(med[med['duration'] == pd.Timedelta(0)]['description'].value_counts().head(8))
print("Longest:")
display(med.nlargest(8, 'duration_days')[['description', 'start', 'stop', 'duration_days']])

# 3. Birth / death (1-day tolerance, patients' dates are date-only)
tol = pd.Timedelta(days=1)
m = med.merge(pt[['id', 'birthdate', 'deathdate']].rename(columns={'id': 'pid'}),
              left_on='patient', right_on='pid', how='left')
dead = m['deathdate'].notna()
print("\nNo matching patient:", m['birthdate'].isna().sum())
print("start < birth (tolerance):", (m['start_dt'] < m['birthdate'] - tol).sum())
print("start > death (strict / tolerance):", (dead & (m['start_dt'] > m['deathdate'])).sum(), "/",
      (dead & (m['start_dt'] > m['deathdate'] + tol)).sum())
print("stop  > death (tolerance):", (dead & (m['stop_dt'] > m['deathdate'] + tol)).sum())

# 4. Belongs to its encounter
e = enc[['id', 'start', 'stop', 'patient', 'encounterclass', 'description']].copy()
e['e_start'] = pd.to_datetime(e['start'], utc=True).dt.tz_localize(None)
e['e_stop']  = pd.to_datetime(e['stop'],  utc=True).dt.tz_localize(None)
e = e.rename(columns={'id': 'eid', 'patient': 'e_patient', 'description': 'enc_description'})
x = m.merge(e[['eid', 'e_start', 'e_stop', 'e_patient', 'encounterclass', 'enc_description']],
            left_on='encounter', right_on='eid', how='left')

print("\nNo matching encounter:", x['eid'].isna().sum())
print("Patient differs from encounter's patient:", (x['patient'] != x['e_patient']).sum())
print("Start before encounter start:", (x['start_dt'] < x['e_start']).sum())
print("Start exactly at encounter start:", (x['start_dt'] == x['e_start']).sum())
late = x[x['start_dt'] > x['e_stop']].copy()
late['days_after_encounter'] = (late['start_dt'] - late['e_stop']).dt.total_seconds() / 86400
print("Start after encounter end:", len(late))
print(late['days_after_encounter'].round(0).value_counts().head(5))
print(late['encounterclass'].value_counts())
print("Stop before encounter start:", (x['stop_dt'] < x['e_start']).sum())

Rows: 8001 | missing stop: 348 (4.3%)
Unparsed start: 0 | unparsed stop (non-null raw): 0

Negative duration: 6
Zero duration:     1614
count     7653.00
mean        92.71
std        503.08
min         -6.00
25%          7.00
50%         12.00
75%         49.00
max      17285.00
Name: duration_days, dtype: float64
0.500      12.0
0.900     371.0
0.950     371.0
0.990     377.0
0.999    8517.1
Name: duration_days, dtype: float64
> 1 days: 6021
> 30 days: 2312
> 365 days: 793
> 3650 days: 19
> 7300 days: 11

Negative durations:


,patient,encounter,description,start,stop,dispenses,totalcost
2926,6b3e2962-5d6f-9834-dca9-124c141da82e,6b3e2962-5d6f-9834-f9be-c7754cc209fc,lisinopril 10 MG Oral Tablet,2022-07-12T18:20:31Z,2022-07-06T18:20:31Z,1,0.91
2927,6b3e2962-5d6f-9834-dca9-124c141da82e,6b3e2962-5d6f-9834-f9be-c7754cc209fc,amLODIPine 2.5 MG Oral Tablet,2022-07-12T18:20:31Z,2022-07-06T18:20:31Z,1,0.91
3899,3c4404e5-2adb-77fb-0fbc-a3cf7bb4179f,3c4404e5-2adb-77fb-ca05-74b4d0dfdc97,Hydrochlorothiazide 25 MG Oral Tablet,2024-01-14T07:00:28Z,2024-01-12T07:00:28Z,1,0.45
3900,3c4404e5-2adb-77fb-0fbc-a3cf7bb4179f,3c4404e5-2adb-77fb-ca05-74b4d0dfdc97,Acetaminophen 325 MG Oral Tablet [Tylenol],2024-01-14T07:00:28Z,2024-01-12T07:00:28Z,3,389.82
3901,3c4404e5-2adb-77fb-0fbc-a3cf7bb4179f,3c4404e5-2adb-77fb-ca05-74b4d0dfdc97,24 HR Metformin hydrochloride 500 MG Extended ...,2024-01-14T07:00:28Z,2024-01-12T07:00:28Z,1,0.91
3902,3c4404e5-2adb-77fb-0fbc-a3cf7bb4179f,3c4404e5-2adb-77fb-ca05-74b4d0dfdc97,lisinopril 10 MG Oral Tablet,2024-01-14T07:00:28Z,2024-01-12T07:00:28Z,1,0.91


Zero-duration medications (top):
description
1 ML Epoetin Alfa 4000 UNT/ML Injection [Epogen]      782
sodium fluoride 0.0272 MG/MG Oral Gel                 416
tropicamide 5 MG/ML Ophthalmic Solution               126
10 ML Furosemide 10 MG/ML Injection                    74
Acetaminophen 500 MG Oral Tablet                       30
1 ML tacrolimus 5 MG/ML Injection [Prograf]            29
1 ML Enoxaparin sodium 150 MG/ML Prefilled Syringe     21
10 ML oxaliplatin 5 MG/ML Injection                    18
Name: count, dtype: int64
Longest:


,description,start,stop,duration_days
4975,Simvastatin 10 MG Oral Tablet,1969-11-04T22:08:49Z,2017-03-02T22:08:49Z,17285.0
48,Simvastatin 10 MG Oral Tablet,1983-10-21T13:52:05Z,2026-06-05T13:52:05Z,15568.0
4809,Simvastatin 10 MG Oral Tablet,1987-03-03T16:53:53Z,2025-09-23T16:53:53Z,14084.0
2723,Simvastatin 10 MG Oral Tablet,1988-04-29T04:38:11Z,2025-11-28T04:38:11Z,13727.0
5931,Simvastatin 10 MG Oral Tablet,1967-10-31T22:08:49Z,1999-06-30T22:08:49Z,11565.0
3097,Simvastatin 10 MG Oral Tablet,1995-10-15T07:00:28Z,2026-04-29T07:00:28Z,11154.0
2825,Simvastatin 10 MG Oral Tablet,1996-06-14T15:07:06Z,2025-10-17T15:07:06Z,10717.0
759,Simvastatin 10 MG Oral Tablet,2000-11-05T12:22:47Z,2026-02-02T12:22:47Z,9220.0



No matching patient: 0
start < birth (tolerance): 0
start > death (strict / tolerance): 6 / 0
stop  > death (tolerance): 0

No matching encounter: 0
Patient differs from encounter's patient: 0
Start before encounter start: 0
Start exactly at encounter start: 6189
Start after encounter end: 81
days_after_encounter
7.0      62
5.0       7
1.0       4
75.0      4
344.0     2
Name: count, dtype: int64
encounterclass
wellness      59
outpatient    10
ambulatory     9
urgentcare     3
Name: count, dtype: int64
Stop before encounter start: 6


In [44]:
med['stop_before_start_flag'] = med['duration'] < pd.Timedelta(0)

### Medications: patient and encounter relationships

All medications reference an existing patient and encounter (0 orphans). Every medication's patient matches its encounter's patient (0 mismatches; merge preserved all 8,001 rows). 116 of 120 patients have medications; 52% of encounters do (urgentcare 93%, 0% for home/hospice/snf). **No changes made.**

In [45]:
med = data['medications']
pt  = data['patients']
enc = data['encounters']

# Clean keys (same as before)
for df, cols in [(pt, ['id']), (enc, ['id', 'patient']), (med, ['patient', 'encounter'])]:
    for c in cols:
        df[c] = df[c].astype(str).str.strip()

print("Null patient / encounter in medications:",
      med['patient'].isna().sum(), "/", med['encounter'].isna().sum())

# 1. medications.patient → patients.id
orph_p = set(med['patient']) - set(pt['id'])
print(f"\nOrphan patient IDs: {len(orph_p)} ({med['patient'].isin(orph_p).sum()} rows)")

# 2. medications.encounter → encounters.id
orph_e = set(med['encounter']) - set(enc['id'])
print(f"Orphan encounter IDs: {len(orph_e)} ({med['encounter'].isin(orph_e).sum()} rows)")

# 3. Patient consistency: medications.patient == encounters[medications.encounter].patient
chk = med.merge(
    enc[['id', 'patient']].rename(columns={'id': 'eid', 'patient': 'enc_patient'}),
    left_on='encounter', right_on='eid', how='left'
)
print(f"\nRows after merge: {len(chk)} (should equal {len(med)})")
mismatch = chk[chk['patient'] != chk['enc_patient']]
print(f"Patient mismatches (medication vs its encounter): {len(mismatch)}")
if len(mismatch):
    display(mismatch[['patient', 'encounter', 'enc_patient', 'description']].head(10))
else:
    print("✓ Every medication's patient matches its encounter's patient.")

# 4. Reverse coverage (informational, not errors)
print(f"\nPatients with medications: {med['patient'].nunique()} / {len(pt)}")
print(f"Encounters with medications: {med['encounter'].nunique()} / {len(enc)}")
print("Share of encounters that have medications, by class:")
print(enc.assign(has=enc['id'].isin(med['encounter'])).groupby('encounterclass')['has'].mean().round(2))
print("\nMedications per patient:"); print(med.groupby('patient').size().describe().round(1))
print("Max medications on one encounter:", med.groupby('encounter').size().max())
print("Patients with no medications:", sorted(set(pt['id']) - set(med['patient'])))

Null patient / encounter in medications: 0 / 0

Orphan patient IDs: 0 (0 rows)
Orphan encounter IDs: 0 (0 rows)

Rows after merge: 8001 (should equal 8001)
Patient mismatches (medication vs its encounter): 0
✓ Every medication's patient matches its encounter's patient.

Patients with medications: 116 / 120
Encounters with medications: 3675 / 7042
Share of encounters that have medications, by class:
encounterclass
ambulatory    0.50
emergency     0.73
home          0.00
hospice       0.00
inpatient     0.33
outpatient    0.65
snf           0.00
urgentcare    0.93
virtual       0.20
wellness      0.33
Name: has, dtype: float64

Medications per patient:
count    116.0
mean      69.0
std      175.5
min        1.0
25%        5.0
50%       13.0
75%       45.2
max      956.0
dtype: float64
Max medications on one encounter: 54
Patients with no medications: ['7d7c30ae-3ed3-f953-824b-42a0c05524ea', 'c1b6697e-6cc1-ea0f-8217-e9eb78896401', 'ca3d0b16-2bc0-6445-47c2-0e23c16ad9e8', 'f19b9cae-0ea7-2d5

### Medications: cost logic

No missing or negative values; `payer_coverage <= totalcost` in all rows; `totalcost = base_cost × dispenses` holds exactly in all 8,001 rows. 2,540 rows (32%) have zero coverage (valid).

Extreme values are real inputs, not errors: 2 Alteplase rows ($21.1M with 203 dispenses, $9.9M with 108) make up 82% of all medication cost. Dispenses accumulate roughly monthly for open-ended prescriptions (corr 0.70 with months open; max 1,032 for a med started in 1940). Base prices look like generator defaults (e.g. 129.94 on 62 different drugs).

**No rows removed**; 15 rows with `totalcost > 100,000` flagged with `cost_outlier_flag`.

In [46]:
med = data['medications']
cost_cols = ['base_cost', 'payer_coverage', 'dispenses', 'totalcost']
tol = 0.01

# 1. Missing / negative / zero
print("Missing:", med[cost_cols].isna().sum().to_dict())
for c in cost_cols:
    print(f"{c}: negative={(med[c] < 0).sum()}, zero={(med[c] == 0).sum()}")
print(med[cost_cols].describe().T.round(2))
print("dispenses non-integer:", (med['dispenses'] != med['dispenses'].round()).sum())
print("cost columns with >2 decimals:", {c: int((med[c].round(2) != med[c]).sum())
                                         for c in ['base_cost', 'payer_coverage', 'totalcost']})

# 2. Coverage cannot exceed the total
print("\npayer_coverage > totalcost:", (med['payer_coverage'] > med['totalcost'] + tol).sum())

# 3. totalcost vs base_cost * dispenses
med['expected_total'] = med['base_cost'] * med['dispenses']
diff = med['totalcost'] - med['expected_total']
print("\nMatches within 1 cent:", (diff.abs() <= tol).sum(), "/", len(med))
print("totalcost > expected:", (diff > tol).sum(), "| totalcost < expected:", (diff < -tol).sum())
ratio = med['totalcost'] / med['expected_total']
print("Ratio totalcost / expected:", ratio.describe().round(3).to_dict())
display(med.loc[diff.abs() > tol,
        ['description', 'base_cost', 'dispenses', 'totalcost', 'expected_total']].head(10))

# 4. Extremes (inspect, do not remove)
print("\nTotalcost quantiles:", med['totalcost'].quantile([.5, .9, .99, .999]).round(1).to_dict())
print("Rows with totalcost > 100,000:", (med['totalcost'] > 1e5).sum(),
      "| > 1,000,000:", (med['totalcost'] > 1e6).sum())
print("Rows with dispenses > 100:", (med['dispenses'] > 100).sum(),
      "| > 1,000:", (med['dispenses'] > 1000).sum())
show = ['description', 'base_cost', 'dispenses', 'totalcost', 'payer_coverage', 'start', 'stop']
display(med.nlargest(8, 'totalcost')[show])
display(med.nlargest(8, 'dispenses')[show])
tot = med['totalcost'].sum()
print(f"Share of all medication cost: top 2 rows {med.nlargest(2,'totalcost')['totalcost'].sum()/tot:.1%}, "
      f"top 10 rows {med.nlargest(10,'totalcost')['totalcost'].sum()/tot:.1%}")

# 5. Why are dispenses so high? Compare with how long the prescription has been open
end = pd.Timestamp('2026-09-23', tz='UTC')   # latest date in the data
s = pd.to_datetime(med['start'], utc=True)
e = pd.to_datetime(med['stop'], utc=True).fillna(end)
med['months_open'] = (e - s).dt.days / 30.44
open_meds = med[med['stop'].isna()]
print("\nOpen medications (no stop):", len(open_meds))
print("corr(dispenses, months open):", round(open_meds['dispenses'].corr(open_meds['months_open']), 2))
print("dispenses per month open:", (open_meds['dispenses'] / open_meds['months_open']).describe().round(2).to_dict())
print("Rows with dispenses > 12 that have no stop:",
      round(med[med['dispenses'] > 12]['stop'].isna().mean(), 2))

# 6. Recurring base prices
print("\nMost common base_cost values:")
print(med['base_cost'].value_counts().head(5))
shared = med[med['base_cost'] == 129.94]
print("Rows at 129.94:", len(shared), "across", shared['description'].nunique(), "different drugs")
print(shared['description'].value_counts().head(5))

Missing: {'base_cost': 0, 'payer_coverage': 0, 'dispenses': 0, 'totalcost': 0}
base_cost: negative=0, zero=0
payer_coverage: negative=0, zero=2540
dispenses: negative=0, zero=0
totalcost: negative=0, zero=0
                 count     mean        std   min   25%   50%     75%  \
base_cost       8001.0   108.08    1864.57  0.01  0.91  4.64  129.94   
payer_coverage  8001.0    69.23    1491.00  0.00  0.00  0.73   23.37   
dispenses       8001.0     6.61      43.40  1.00  1.00  1.00    1.00   
totalcost       8001.0  4723.80  260749.62  0.01  0.91  6.25  129.94   

                        max  
base_cost         103958.40  
payer_coverage     83166.72  
dispenses           1032.00  
totalcost       21103555.20  
dispenses non-integer: 0
cost columns with >2 decimals: {'base_cost': 0, 'payer_coverage': 0, 'totalcost': 0}

payer_coverage > totalcost: 0

Matches within 1 cent: 8001 / 8001
totalcost > expected: 0 | totalcost < expected: 0
Ratio totalcost / expected: {'count': 8001.0, 'mean': 1

,description,base_cost,dispenses,totalcost,expected_total



Totalcost quantiles: {0.5: 6.2, 0.9: 389.8, 0.99: 20180.1, 0.999: 123954.8}
Rows with totalcost > 100,000: 15 | > 1,000,000: 2
Rows with dispenses > 100: 110 | > 1,000: 1


,description,base_cost,dispenses,totalcost,payer_coverage,start,stop
4980,Alteplase 100 MG Injection,103958.40,203,21103555.20,83166.72,2001-07-31T22:08:49Z,NaN
4701,Alteplase 100 MG Injection,91728.00,108,9906624.00,73348.10,2017-10-24T05:01:28Z,NaN
4978,Donepezil hydrochloride 10 MG Oral Tablet,2154.17,194,417908.98,1723.34,2000-01-11T22:08:49Z,2015-12-26T22:08:49Z
4971,Memantine hydrochloride 2 MG/ML Oral Solution,3759.03,85,319517.55,3007.22,2006-09-16T22:08:49Z,NaN
3981,Clopidogrel 75 MG Oral Tablet,683.16,209,142780.44,633.16,2009-07-23T19:24:33Z,NaN
3543,diphenhydrAMINE Hydrochloride 25 MG Oral Tablet,163.55,842,137709.10,81.32,1957-07-25T00:19:06Z,NaN
4810,Clopidogrel 75 MG Oral Tablet,463.48,285,132091.80,0.00,2003-04-29T16:53:53Z,NaN
1292,Clopidogrel 75 MG Oral Tablet,507.41,257,130404.37,457.41,1998-05-22T15:07:06Z,NaN


,description,base_cost,dispenses,totalcost,payer_coverage,start,stop
7649,Acetaminophen 325 MG / oxyCODONE Hydrochloride...,7.41,1032,7647.12,0.00,1940-10-15T22:46:11Z,NaN
6835,Meperidine Hydrochloride 50 MG Oral Tablet,87.62,983,86130.46,0.00,1948-06-14T23:17:12Z,NaN
3543,diphenhydrAMINE Hydrochloride 25 MG Oral Tablet,163.55,842,137709.10,81.32,1957-07-25T00:19:06Z,NaN
529,diphenhydrAMINE Hydrochloride 25 MG Oral Tablet,61.15,829,50693.35,0.00,1958-08-11T17:30:14Z,NaN
3545,doxycycline hyclate 100 MG,129.94,792,102912.48,103.95,1961-09-02T00:19:06Z,NaN
4808,Vitamin B12 5 MG/ML Injectable Solution,5.04,756,3810.24,0.00,1964-07-28T18:03:31Z,NaN
4172,Chlorpheniramine Maleate 2 MG/ML Oral Solution,78.89,733,57826.37,0.00,1966-07-01T08:55:05Z,NaN
541,Chlorpheniramine Maleate 2 MG/ML Oral Solution,152.88,699,106863.12,0.00,1969-04-09T01:39:17Z,NaN


Share of all medication cost: top 2 rows 82.0%, top 10 rows 86.1%

Open medications (no stop): 348
corr(dispenses, months open): 0.7
dispenses per month open: {'count': 348.0, 'mean': 1.39, 'std': 3.67, 'min': 0.0, '25%': 0.25, '50%': 0.95, '75%': 1.01, 'max': 45.66}
Rows with dispenses > 12 that have no stop: 0.91

Most common base_cost values:
base_cost
0.91      1817
129.94    1407
0.45       677
29.21      485
30.67      281
Name: count, dtype: int64
Rows at 129.94: 1407 across 62 different drugs
description
sodium fluoride 0.0272 MG/MG Oral Gel          416
Acetaminophen 325 MG Oral Tablet [Tylenol]     171
tropicamide 5 MG/ML Ophthalmic Solution        126
Ibuprofen 400 MG Oral Tablet [Ibu]             100
budesonide 0.25 MG/ML Inhalation Suspension     50
Name: count, dtype: int64


In [47]:
med['cost_outlier_flag'] = (med['totalcost'] > 100_000)   # 15 rows

### Medications: code ↔ description

No nulls; codes are numeric (6-7 digits), stored as text. 153 codes ↔ 155 descriptions: 2 codes had two descriptions differing only in capitalization (medroxyprogesterone, aspirin); standardized to the most frequent variant per code. Descriptions → codes is 1:1. Repeated internal spaces (647 rows) collapsed.

8 pairs share patient+encounter+code+start (single fill + full course); kept and the single-fill row flagged with `possible_duplicate_flag`. **No rows removed.**

In [48]:
raw = pd.read_csv(base_url + "medications.csv", dtype=str)   # code as text
raw.columns = [c.lower() for c in raw.columns]
med = data['medications']

# 1. Missing, numeric, whitespace, lengths
s = raw['code']
print("Null code / description:", med['code'].isna().sum(), "/", med['description'].isna().sum())
print("Non-numeric:", (~s.str.fullmatch(r'\d+', na=False)).sum(),
      "| whitespace:", (s != s.str.strip()).sum(), "| leading zeros:", s.str.startswith('0').sum())
print(s.str.len().value_counts().sort_index())

# 2. Mapping
print("\nDistinct codes:", med['code'].nunique(),
      "| descriptions:", med['description'].nunique(),
      "| pairs:", med[['code', 'description']].drop_duplicates().shape[0])
per_code = med.groupby('code')['description'].nunique()
per_desc = med.groupby('description')['code'].nunique()
print("Codes with >1 description:", (per_code > 1).sum())
print("Descriptions with >1 code:", (per_desc > 1).sum())
pairs = med.groupby(['code', 'description']).size().reset_index(name='rows')
display(pairs[pairs['code'].isin(per_code[per_code > 1].index)])
display(pairs[pairs['description'].isin(per_desc[per_desc > 1].index)])

# 3. Are the "different" descriptions just formatting? Normalize and re-check
norm = med['description'].str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
print("\nDistinct descriptions after lowercase + collapsing spaces:", norm.nunique(),
      "(codes:", med['code'].nunique(), ")")
d = med['description']
print("Whitespace at ends:", (d != d.str.strip()).sum(),
      "| rows with double spaces:", d.str.contains('  ').sum(),
      "| distinct such descriptions:", d[d.str.contains('  ')].nunique())

# 4. Duplicates
print("\nExact duplicate rows:", med.duplicated().sum())
key = ['patient', 'encounter', 'code', 'start']
dup = med[med.duplicated(key, keep=False)].sort_values(key)
print("Rows sharing patient+encounter+code+start:", len(dup), f"({len(dup)//2} pairs)")
display(dup[['patient', 'code', 'description', 'start', 'stop', 'base_cost',
             'dispenses', 'totalcost']])

Null code / description: 0 / 0
Non-numeric: 0 | whitespace: 0 | leading zeros: 0
code
6    7203
7     798
Name: count, dtype: int64

Distinct codes: 153 | descriptions: 155 | pairs: 155
Codes with >1 description: 2
Descriptions with >1 code: 0


,code,description,rows
24,243670,Aspirin 81 MG Oral Tablet,2
25,243670,aspirin 81 MG Oral Tablet,9
109,1000126,1 ML medroxyPROGESTERone acetate 150 MG/ML Inj...,1
110,1000126,1 ML medroxyprogesterone acetate 150 MG/ML Inj...,10


,code,description,rows



Distinct descriptions after lowercase + collapsing spaces: 153 (codes: 153 )
Whitespace at ends: 0 | rows with double spaces: 647 | distinct such descriptions: 6

Exact duplicate rows: 0
Rows sharing patient+encounter+code+start: 16 (8 pairs)


,patient,code,description,start,stop,base_cost,dispenses,totalcost
107,2f72e25c-fbfa-4b6d-8061-c74714cfe7c9,904419,Alendronic acid 10 MG Oral Tablet,2017-09-21T05:26:55Z,2017-09-21T05:26:55Z,5.70,1,5.70
108,2f72e25c-fbfa-4b6d-8061-c74714cfe7c9,904419,Alendronic acid 10 MG Oral Tablet,2017-09-21T05:26:55Z,2018-09-27T05:26:55Z,2.11,12,25.32
128,365dc87a-536c-ee7e-c7bf-895a948b921b,310798,Hydrochlorothiazide 25 MG Oral Tablet,2019-10-03T09:41:25Z,2019-10-03T09:41:25Z,0.45,1,0.45
129,365dc87a-536c-ee7e-c7bf-895a948b921b,310798,Hydrochlorothiazide 25 MG Oral Tablet,2019-10-03T09:41:25Z,2020-10-08T09:41:25Z,0.45,4,1.80
892,8acba407-e36b-b38f-1bc3-57bd83578001,904419,Alendronic acid 10 MG Oral Tablet,2017-05-23T23:57:26Z,2017-05-23T23:57:26Z,11.66,1,11.66
893,8acba407-e36b-b38f-1bc3-57bd83578001,904419,Alendronic acid 10 MG Oral Tablet,2017-05-23T23:57:26Z,2018-05-29T23:57:26Z,5.19,12,62.28
843,abf6144c-7ddb-2559-c3ca-20d07a1b2989,860975,24 HR Metformin hydrochloride 500 MG Extended ...,2021-07-23T23:14:54Z,2021-07-23T23:14:54Z,0.15,1,0.15
844,abf6144c-7ddb-2559-c3ca-20d07a1b2989,860975,24 HR Metformin hydrochloride 500 MG Extended ...,2021-07-23T23:14:54Z,2022-07-29T23:14:54Z,2.28,12,27.36
3032,cb1678fa-98d5-1d6b-ee5b-94a8fffb5040,106892,insulin isophane human 70 UNT/ML / insulin r...,1988-12-22T19:24:33Z,1988-12-22T19:24:33Z,507.86,1,507.86
3033,cb1678fa-98d5-1d6b-ee5b-94a8fffb5040,106892,insulin isophane human 70 UNT/ML / insulin r...,1988-12-22T19:24:33Z,1989-05-25T19:24:33Z,126.89,5,634.45


In [49]:
# Collapse repeated spaces first, then use one description per code
med['description'] = med['description'].str.replace(r'\s+', ' ', regex=True).str.strip()
canon = med.groupby('code')['description'].agg(lambda s: s.value_counts().idxmax())
med['description'] = med['code'].map(canon)

print("Codes with >1 description now:", (med.groupby('code')['description'].nunique() > 1).sum())
data['medications']['code'] = data['medications']['code'].astype(str)   # store as text

Codes with >1 description now: 0


In [50]:
med['possible_duplicate_flag'] = (
    med.duplicated(key, keep=False) & (med['start'] == med['stop'])
)
print(med['possible_duplicate_flag'].sum())   # 8

8


### Medications: reason fields

`reasoncode` and `reasondescription` are missing together in exactly the same 1,047 rows (13.1%); no partial records, no blanks, strict 1:1 code↔description mapping (38 codes). Missingness is driven by the drug (93 of 155 drugs never have a reason, 58 always do): NULL = no indication recorded (OTC/one-off drugs).

Medication reason differs from encounter reason (match only 54%) and matches a condition of the same patient in 91% of rows (rest: dental referral procedures, anemia, asthma). Kept as NULL; `reasoncode` stored as text. **No rows removed.**

In [51]:
med = data['medications']
enc = data['encounters']
con = data['conditions']

# Read reasoncode as text so long codes and NaNs don't turn into floats
med['reasoncode'] = med['reasoncode'].astype('string').str.replace(r'\.0$', '', regex=True)

rc_null = med['reasoncode'].isna()
rd_null = med['reasondescription'].isna()

# 1. Do they go missing together?
print(f"reasoncode null: {rc_null.sum()} ({rc_null.mean():.1%}) | reasondescription null: {rd_null.sum()}")
print(pd.crosstab(rc_null.rename('code_null'), rd_null.rename('desc_null')))
print("Code missing, description present:", (rc_null & ~rd_null).sum())
print("Code present, description missing:", (~rc_null & rd_null).sum())

# 2. Format
v = med['reasondescription'].dropna()
print("\nEmpty strings:", (v.str.strip() == '').sum(), "| whitespace:", (v != v.str.strip()).sum(),
      "| double spaces:", v.str.contains('  ').sum())
s = med['reasoncode'].dropna()
print("Numeric:", s.str.fullmatch(r'\d+').all(), "| lengths:", s.str.len().value_counts().sort_index().to_dict())

# 3. 1:1 mapping
has = med.dropna(subset=['reasoncode'])
print("Distinct reason codes:", has['reasoncode'].nunique())
print("Descriptions per code (max):", has.groupby('reasoncode')['reasondescription'].nunique().max())
print("Codes per description (max):", has.groupby('reasondescription')['reasoncode'].nunique().max())

# 4. Is missingness tied to the drug?
by_drug = med.groupby('description').agg(n=('code', 'size'), null_share=('reasoncode', lambda x: x.isna().mean()))
print(f"\nDrugs: {len(by_drug)} | always null: {(by_drug['null_share'] == 1).sum()} | "
      f"never null: {(by_drug['null_share'] == 0).sum()} | mixed: {by_drug['null_share'].between(0.01, 0.99).sum()}")
print(by_drug[by_drug['null_share'] == 1].sort_values('n', ascending=False).head(8))
print(by_drug[by_drug['null_share'].between(0.01, 0.99)].sort_values('n', ascending=False))

# 5. Versus the encounter's reason
e = enc[['id', 'encounterclass', 'reasoncode']].rename(columns={'id': 'eid', 'reasoncode': 'enc_reasoncode'})
e['enc_reasoncode'] = e['enc_reasoncode'].astype('string').str.replace(r'\.0$', '', regex=True)
m = med.merge(e, left_on='encounter', right_on='eid', how='left')
print("\nMedication reason null, by encounter class:")
print(m.groupby('encounterclass')['reasoncode'].apply(lambda x: x.isna().mean()).round(2))
print(pd.crosstab(m['reasoncode'].isna().rename('med_reason_null'), m['enc_reasoncode'].isna().rename('enc_reason_null')))
both = m.dropna(subset=['reasoncode', 'enc_reasoncode'])
print("Med reason == encounter reason (where both exist):", round((both['reasoncode'] == both['enc_reasoncode']).mean(), 3), f"({len(both)} rows)")

# 6. Is the reason a condition the same patient actually has?
con_pairs = set(zip(con['patient'], con['code'].astype(str)))
has = has.assign(in_conditions=[(p, c) in con_pairs for p, c in zip(has['patient'], has['reasoncode'])])
print("\nReason matches a condition code of the same patient:", round(has['in_conditions'].mean(), 3))
print(has.loc[~has['in_conditions'], 'reasondescription'].value_counts().head(6))

reasoncode null: 1047 (13.1%) | reasondescription null: 1047
desc_null  False  True 
code_null              
False       6954      0
True           0   1047
Code missing, description present: 0
Code present, description missing: 0

Empty strings: 0 | whitespace: 0 | double spaces: 3
Numeric: True | lengths: {np.int64(7): 8, np.int64(8): 5383, np.int64(9): 1556, np.int64(15): 7}
Distinct reason codes: 38
Descriptions per code (max): 1
Codes per description (max): 1

Drugs: 153 | always null: 91 | never null: 58 | mixed: 4
                                                      n  null_share
description                                                        
Acetaminophen 325 MG Oral Tablet [Tylenol]          171         1.0
tropicamide 5 MG/ML Ophthalmic Solution             126         1.0
Ibuprofen 400 MG Oral Tablet [Ibu]                  100         1.0
Acetaminophen 300 MG / Hydrocodone Bitartrate 5...   77         1.0
Acetaminophen 500 MG Oral Tablet                     30         1

In [52]:
med = data['medications']
enc = data['encounters']

def to_text_code(s):
    # float/object codes -> clean text; NaN stays missing (becomes NULL on upload)
    return pd.to_numeric(s, errors='coerce').astype('Int64').astype('string')

# reason codes as text (medications and encounters, so they match)
med['reasoncode'] = to_text_code(med['reasoncode'])
enc['reasoncode'] = to_text_code(enc['reasoncode'])

# collapse repeated spaces in the reason description (NaN is left alone)
med['reasondescription'] = (med['reasondescription']
                            .str.replace(r'\s+', ' ', regex=True)
                            .str.strip())

# code columns as text too (conditions, medications)
data['conditions']['code'] = data['conditions']['code'].astype('string')
med['code'] = med['code'].astype('string')

# verify
print(med['reasoncode'].dtype, med['reasoncode'].isna().sum())          # string, 1047
print(med['reasoncode'].dropna().head(3).tolist())                       # e.g. ['59621000', ...] with no '.0'
print("double spaces left:", med['reasondescription'].str.contains('  ', na=False).sum())  # 0
print("null alignment kept:", (med['reasoncode'].isna() == med['reasondescription'].isna()).all())  # True

string 1047
['10509002', '103697008', '103697008']
double spaces left: 0
null alignment kept: True


### Observations: type

`type` has only `numeric` (70,767) and `text` (46,735); no nulls, whitespace or case variants. All numeric values parse as floats. 4 descriptions typed `text` always contain numbers (family income, LogMAR left/right, Priority Level; 1,787 rows) and were reclassified to `numeric`.

Split into `value_numeric` (float) and `value_text`. 7 urine-test descriptions appear under both types (119 numeric `{presence}` rows vs text results) and were left as is. Text rows carry placeholder unit `{nominal}`. **No rows removed.**

In [53]:
obs = data['observations']

# 1. Validate TYPE
t = obs['type']
print("Null types:", t.isna().sum())
print(t.value_counts(dropna=False))
v = t.dropna().astype(str)
print("Whitespace:", (v != v.str.strip()).sum(),
      "| empty:", (v.str.strip() == '').sum(),
      "| not lowercase:", (v != v.str.lower()).sum())
allowed = {'numeric', 'text'}
print("Unexpected types:", set(v.unique()) - allowed or "none")

# 2. Split logic: does VALUE agree with TYPE?
obs['value_parsed'] = pd.to_numeric(obs['value'], errors='coerce')
num = obs[obs['type'] == 'numeric']
txt = obs[obs['type'] == 'text']
print(f"\nNumeric rows: {len(num)} | value unparsable as number: {num['value_parsed'].isna().sum()} | null value: {num['value'].isna().sum()}")
print(f"Text rows:    {len(txt)} | value parses as a number: {txt['value_parsed'].notna().sum()} | null value: {txt['value'].isna().sum()}")

# 3. Which text rows are really numbers?
g = txt.groupby('description').agg(n=('value_parsed', 'size'),
                                   numeric_share=('value_parsed', lambda s: s.notna().mean()))
display(g[g['numeric_share'] > 0].sort_values('n', ascending=False))

# 4. Units by type
print("\nText rows with units:", txt['units'].notna().sum(), "of", len(txt))
print(txt['units'].value_counts().head(8))
print("Numeric rows with no units:", num['units'].isna().sum())
display(num[num['units'].isna()][['category', 'description', 'value']])

# 5. Same description recorded under both types
types_per_desc = obs.groupby('description')['type'].nunique()
mixed = types_per_desc[types_per_desc > 1].index
print("\nDescriptions with both types:", len(mixed))
display(obs[obs['description'].isin(mixed)].groupby(['description', 'type']).size().unstack(fill_value=0))

# 6. Type by category
print(pd.crosstab(obs['category'].fillna('(missing)'), obs['type']))

Null types: 0
type
numeric    70767
text       46735
Name: count, dtype: int64
Whitespace: 0 | empty: 0 | not lowercase: 0
Unexpected types: none

Numeric rows: 70767 | value unparsable as number: 0 | null value: 0
Text rows:    46735 | value parses as a number: 1787 | null value: 0


,n,numeric_share
description,,
What was your best estimate of the total income of all family members from all sources before taxes in last year [PhenX],1525,1.0
LogMAR visual acuity left eye (observable entity),126,1.0
LogMAR visual acuity right eye (observable entity),126,1.0
Priority Level,10,1.0



Text rows with units: 14890 of 46735
units
{nominal}    13087
/a            1525
{logmar}       252
{score}         20
{#}              6
Name: count, dtype: int64
Numeric rows with no units: 3


,category,description,value
22267,exam,Position of body and posture (observable entity),1.0
22268,exam,Cobb angle (observable entity),11.3
22269,exam,Risser sign (finding),0.0



Descriptions with both types: 7


type,numeric,text
description,,
Bilirubin.total [Presence] in Urine by Test strip,17,1174
Glucose [Presence] in Urine by Test strip,17,1174
Hemoglobin [Presence] in Urine by Test strip,17,1183
Ketones [Presence] in Urine by Test strip,17,1183
Leukocyte esterase [Presence] in Urine by Test strip,17,1183
Nitrite [Presence] in Urine by Test strip,17,1183
Protein [Presence] in Urine by Test strip,17,1174


type            numeric   text
category                      
(missing)          3189      0
exam                  7    552
imaging               0    100
laboratory        42920  13086
procedure            82     10
social-history        0   1956
survey             5718  31027
therapy               0      4
vital-signs       18851      0


In [54]:
# Reclassify descriptions whose text values are always numeric
text_rows = obs['type'] == 'text'
always_num = (obs[text_rows].groupby('description')['value_parsed']
              .apply(lambda s: s.notna().all()))
promote = always_num[always_num].index
print("Descriptions to reclassify:", list(promote))
obs.loc[text_rows & obs['description'].isin(promote), 'type'] = 'numeric'

# Two typed value columns: numeric rows get a float, text rows keep the string
obs['value_numeric'] = np.where(obs['type'] == 'numeric', obs['value_parsed'], np.nan)
obs['value_text']    = np.where(obs['type'] == 'text', obs['value'], None)
obs = obs.drop(columns=['value_parsed'])

print(obs['type'].value_counts())                     # numeric 72,554 / text 44,948
print("numeric rows with no value_numeric:",
      ((obs['type'] == 'numeric') & obs['value_numeric'].isna()).sum())   # 0
print("text rows with no value_text:",
      ((obs['type'] == 'text') & obs['value_text'].isna()).sum())          # 0

Descriptions to reclassify: ['LogMAR visual acuity left eye (observable entity)', 'LogMAR visual acuity right eye (observable entity)', 'Priority Level', 'What was your best estimate of the total income of all family members from all sources  before taxes  in last year [PhenX]']
type
numeric    72554
text       44948
Name: count, dtype: int64
numeric rows with no value_numeric: 0
text rows with no value_text: 0


### Observations: numeric values

All 72,554 numeric rows convert to numbers (strict plain-number test: 0 failures; no NaN/inf, scientific notation, commas or padding). All values have at most 1 decimal place.

Flagged, not removed (`suspect_value_flag`): one impossible LDL of -18 mg/dL, and urine specific gravity (1,200 rows, all 1.0 because of one-decimal rounding). Zeros (2,237) are mostly valid questionnaire scores and DALY/QALY. Other negatives are DXA T-scores (valid). Stored in `value_numeric` (float).

In [55]:
obs = data['observations']
num = obs[obs['type'] == 'numeric'].copy()

# 1. Lenient vs strict conversion
num['value_parsed'] = pd.to_numeric(num['value'], errors='coerce')
strict = num['value'].str.fullmatch(r'-?\d+(\.\d+)?')       # plain decimal numbers only
print("Numeric rows:", len(num))
print("Lenient parse fails:", num['value_parsed'].isna().sum())
print("Fail strict plain-number format:", (~strict).sum())
display(num.loc[~strict, ['category', 'description', 'value', 'units']].head(20))
print(num.loc[~strict, 'value'].value_counts().head(10))

# 2. What the lenient parser would have accepted
print("\nNull / blank / padded with spaces:",
      num['value'].isna().sum(), "/", (num['value'].str.strip() == '').sum(), "/",
      (num['value'] != num['value'].str.strip()).sum())
print("inf or NaN:", np.isinf(num['value_parsed']).sum(), "/", num['value_parsed'].isna().sum())
print("Scientific notation / leading '+' / commas:",
      num['value'].str.contains('[eE]').sum(), "/",
      num['value'].str.startswith('+').sum(), "/",
      num['value'].str.contains(',').sum())

# 3. Range and shape
print("\n", num['value_parsed'].describe().round(2))
dec = num['value'].str.extract(r'\.(\d+)$')[0].str.len()
print("Max decimal places:", dec.max())
print("Negative:", (num['value_parsed'] < 0).sum(), "| zero:", (num['value_parsed'] == 0).sum())
display(num[num['value_parsed'] < 0][['patient', 'date', 'description', 'value', 'units']])
print(num[num['value_parsed'] == 0]['description'].value_counts().head(12))

# 4. Descriptions where every value is the same (no information)
spread = num.groupby('description')['value_parsed'].agg(['count', 'nunique', 'min', 'max'])
display(spread[(spread['nunique'] == 1) & (spread['count'] > 20)])

Numeric rows: 72554
Lenient parse fails: 0
Fail strict plain-number format: 0


,category,description,value,units


Series([], Name: count, dtype: int64)

Null / blank / padded with spaces: 0 / 0 / 0
inf or NaN: 0 / 0
Scientific notation / leading '+' / commas: 0 / 0 / 0

 count     72554.00
mean       3116.20
std       34502.14
min         -18.00
25%           4.00
50%          18.90
75%          86.70
max      981530.00
Name: value_parsed, dtype: float64
Max decimal places: 1.0
Negative: 4 | zero: 2237


,patient,date,description,value,units
3307,2f72e25c-fbfa-4b6d-8061-c74714cfe7c9,2017-09-21T05:26:55Z,DXA Radius and Ulna [T-score] Bone density,-3.0,{T-score}
18466,8acba407-e36b-b38f-1bc3-57bd83578001,2017-05-23T23:57:26Z,DXA Radius and Ulna [T-score] Bone density,-3.6,{T-score}
48480,6b3e2962-5d6f-9834-dca9-124c141da82e,2025-07-29T18:20:31Z,Cholesterol in LDL [Mass/volume] in Serum or P...,-18.0,mg/dL
88556,10231522-7925-43d2-2064-bb5f917c87ee,2011-09-20T22:08:49Z,DXA Radius and Ulna [T-score] Bone density,-3.6,{T-score}


description
Total score [HARK]                                                                453
DALY                                                                              446
Patient Health Questionnaire 2 item (PHQ-2) total score [Reported]                297
Pain severity - 0-10 verbal numeric rating [Score] - Reported                     248
Total score [DAST-10]                                                             129
LogMAR visual acuity right eye (observable entity)                                126
LogMAR visual acuity left eye (observable entity)                                 126
Generalized anxiety disorder 7 item (GAD-7) total score [Reported.PHQ]            108
Total score [AUDIT-C]                                                              91
Troponin I.cardiac [Mass/volume] in Serum or Plasma by High sensitivity method     49
QALY                                                                               27
Hemoglobin [Presence] in Urine by Test str

,count,nunique,min,max
description,,,,
LogMAR visual acuity left eye (observable entity),126,1,0.0,0.0
LogMAR visual acuity right eye (observable entity),126,1,0.0,0.0
Specific gravity of Urine by Test strip,1200,1,1.0,1.0


In [56]:
# Convert to a real float column (all rows parse, so no errors)
obs['value_numeric'] = np.where(obs['type'] == 'numeric',
                                pd.to_numeric(obs['value'], errors='coerce'), np.nan)

# Flag suspicious values; keep the rows
ldl_neg = obs['description'].str.contains('Cholesterol in LDL', na=False) & (obs['value_numeric'] < 0)
sg      = obs['description'].eq('Specific gravity of Urine by Test strip') & (obs['type'] == 'numeric')
obs['suspect_value_flag'] = ldl_neg | sg
print(obs['suspect_value_flag'].sum())      # 1 + 1,200 = 1,201

1201
